# Swasthya Medical AI — Final Training Notebook
## Bio_ClinicalBERT + T5 CoT · Disease Prediction + Risk Factors · Kaggle T4 · ~7.5 hrs total

### What this notebook does
Trains two models that together power the full Swasthya clinical AI pipeline:
- **Bio_ClinicalBERT** — 6-head model predicting:
  - Specialist (12 classes)
  - Urgency (4 classes)
  - Severity score (1–10)
  - **Disease prediction — Top-3 with confidence** *(new)*
  - **Risk factor detection — 10 multi-label categories* *(new)*
- **T5 CoT** — generates diagnostic follow-up questions with chain-of-thought reasoning

### Disease prediction approach (3-layer hybrid)
| Layer | Method | Purpose |
|-------|--------|---------|
| **Disease head** | BERT fine-tuned on Kaggle + HCM | AI-powered top-3 disease prediction |
| **Risk factor head** | BERT with weak labels from HCM | Learned multi-label risk extraction |
| **Rule engine** | Regex on raw symptom text | High-precision explicit risk detection |
| **Knowledge map** | Static DISEASE_RISK_MAP dict | Clinically accurate linked risk factors |

### Training stages
| Stage | What trains | BERT | Time |
|-------|------------|------|------|
| Stage 1 | specialist_head only | Frozen | ~24 min |
| Stage 2 | urgency_head only | Frozen | ~24 min |
| Stage 3 | All heads joint | Unfrozen | ~3.5 hrs |
| **Stage 4** | disease_head + risk_head | Frozen | **~15 min** |
| T5 CoT | T5 seq2seq | N/A | ~2 hrs |
| **Total** | | | **~7.5 hrs** |

### New cells added (Part 1B — Disease + Risk)
- **Cell 5C** — Load Kaggle symptom-disease dataset, convert binary → text
- **Cell 5D** — Auto-label HealthCareMagic data with 10 risk factor categories
- **Cell 5E** — Build DISEASE_RISK_MAP (41 diseases → clinical risk factors)
- **Cell 6B** — Extended MedDataset with disease + risk_vector labels
- **Cell 7B** — ClinicalBERTModelExtended with disease_head + risk_head
- **Cell 12B** — Stage 4 training (disease + risk heads, BERT frozen)
- **Cell 13B** — Save extended model + per-disease accuracy report
- **Cell 14B** — Full inference: all 6 outputs in one call

### How to run
1. Enable internet: Settings → Internet → ON
2. No dataset upload needed — Cell 5C auto-downloads from HuggingFace
3. Run Cell 0A → Cell 0B (restart) → Cell 0C
4. Run Cells 1–14 for original BioClinicalBERT training (Stages 1–3)
5. Run Cells 5C–14B for disease + risk factor extension (Stage 4)
6. Run Cells 15–21 for T5 CoT training
7. Download output files from Kaggle Output tab

### Output files (download from Kaggle Output tab)
Copy all to `ml_models/models/` on your machine:
- `biobert_clinical_extended.pth`  ← main model (all 6 heads)
- `disease_encoder.pkl`
- `urgency_encoder.pkl`
- `specialist_encoder.pkl`
- `t5_qgen_model/`                 (entire folder)

### If session is interrupted
Re-run Cells 0C, 1–9, then jump to Stage 4 (Cell 12B) directly.
Stage 4 auto-loads Stage 3 best weights and starts fresh disease training.


## Cell 0A — Install PyTorch 2.3.1
**Run this first. Then run Cell 0B to restart the kernel.**

Kaggle now ships PyTorch 2.10 which has a broken `_utils` module that prevents
Bio_ClinicalBERT from loading. PyTorch 2.3.1 + cu118 is stable and fully compatible.

In [ ]:
!pip install -q torch==2.3.1 torchvision==0.18.1 --index-url https://download.pytorch.org/whl/cu118
print('PyTorch 2.3.1 installed. Run Cell 0B to restart kernel.')

## Cell 0B — Restart Kernel
**Run immediately after Cell 0A.** Required to load the newly installed PyTorch.

In [ ]:
import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)

## Cell 0C — Verify Environment + Install Remaining Packages
**Run this after kernel restarts (and every session after that).**
Cell 0A and 0B only need to run once. From the second session onwards, start here.

In [ ]:
import torch, torch._utils

# Verify PyTorch is healthy before installing anything else
assert hasattr(torch, '_utils'), 'PyTorch _utils missing — re-run Cell 0A and 0B'
print(f'PyTorch : {torch.__version__}  _utils OK: {hasattr(torch, "_utils")}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# Install remaining packages — torch already installed so skip it here
import subprocess
subprocess.run(['pip', 'install', '-q',
    'transformers==4.41.2', 'datasets', 'scikit-learn', 'tqdm'], check=True)

# Clear corrupted HuggingFace cache if present from previous failed downloads
import shutil, os
cache = '/root/.cache/huggingface/hub/models--emilyalsentzer--Bio_ClinicalBERT'
if os.path.exists(cache):
    shutil.rmtree(cache)
    print('Cleared Bio_ClinicalBERT cache — will re-download cleanly in Cell 9')

# Storage path — using /kaggle/working directly for reliable persistence
DRIVE_DIR = '/kaggle/working'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Storage : {DRIVE_DIR}')

# Show what checkpoint files exist from previous sessions
ckpt_files = [f for f in os.listdir(DRIVE_DIR) if f.endswith('.pth') or f.endswith('.pkl')]
if ckpt_files:
    print('\nExisting checkpoint files:')
    for f in sorted(ckpt_files):
        size = os.path.getsize(os.path.join(DRIVE_DIR, f)) / 1e6
        print(f'  {size:6.1f}MB  {f}')
else:
    print('\nNo checkpoint files found — fresh training run')

---
# Part 1 — Bio_ClinicalBERT Training

## Cell 1 — Imports & Config
All imports in one place. `BASE_MODEL` is Bio_ClinicalBERT — pre-trained on MIMIC-III
clinical notes, so it already understands medical terminology before fine-tuning.

In [ ]:
import os, json, pickle, re
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from datasets import load_dataset
from tqdm import tqdm

BASE_MODEL = 'emilyalsentzer/Bio_ClinicalBERT'
DRIVE_DIR  = '/kaggle/working'
device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device    : {device}')
print(f'Model     : {BASE_MODEL}')
print(f'Storage   : {DRIVE_DIR}')

## Cell 2 — NLP Extraction from Doctor Responses (Strategy 1)

Labels are extracted from **what the doctor wrote**, not inferred from patient keywords.
Each function returns `None` when the signal is ambiguous — triggering keyword fallback.

**Negation guard:** 'not urgent' will not match 'urgent'. Each phrase is checked
individually against the 20 characters before it for negation words.

**Severity short-circuit:** `EXPLICIT_LOW_SEVERITY` is checked first so 'not urgent'
always wins over 'i recommend consulting' when both appear in the same response.

All 5 sanity checks must pass before the cell completes — an assert stops execution
if any check fails so you never proceed with broken extraction logic.

In [ ]:
SPECIALIST_PATTERNS = [
    (r'\b(cardiologist|cardiac specialist|heart specialist|cardiac surgeon)\b',         'Cardiologist'),
    (r'\b(pulmonologist|pulmonary specialist|chest physician|respiratory specialist)\b', 'Pulmonologist'),
    (r'\b(neurologist|neurosurgeon|neurological specialist|neuro specialist)\b',         'Neurologist'),
    (r'\b(gastroenterologist|gi specialist|gastrointestinal specialist|hepatologist)\b', 'Gastroenterologist'),
    (r'\b(orthopedic|orthopaedic|orthopedist|bone specialist|joint specialist)\b',       'Orthopedist'),
    (r'\b(dermatologist|skin specialist|skin doctor)\b',                                 'Dermatologist'),
    (r'\b(psychiatrist|mental health specialist|psychologist|counselor|therapist)\b',    'Psychiatrist'),
    (r'\b(urologist|urology specialist|nephrologist|kidney specialist|renal specialist)\b', 'Urologist'),
    (r'\b(gynecologist|gynaecologist|obstetrician|obs and gynae)\b',                    'Gynecologist'),
    (r'\b(ophthalmologist|eye specialist|eye doctor|optometrist)\b',                    'Ophthalmologist'),
    (r'\b(ent specialist|otolaryngologist|ear nose throat|ent doctor|ear specialist)\b', 'ENT Specialist'),
    (r'\b(endocrinologist|diabetes specialist|thyroid specialist|hormone specialist)\b', 'Urologist'),
    (r'\b(general physician|gp|family doctor|primary care|general practitioner)\b',     'General Physician'),
]

def extract_specialist(doctor_text):
    """Extract specialist from doctor's explicit referral. Returns None if not named."""
    if not doctor_text: return None
    t = doctor_text.lower()
    for pattern, specialist in SPECIALIST_PATTERNS:
        if re.search(pattern, t): return specialist
    return None

IMMEDIATE_PHRASES = [
    'immediately', 'right now', 'without delay', 'do not wait',
    'emergency', 'go to the er', 'go to the emergency', 'emergency room',
    'call 911', 'call an ambulance', 'rush to hospital', 'rush to the hospital',
    'life threatening', 'life-threatening', 'cardiac emergency',
    'stroke symptoms', 'heart attack', 'critical condition',
    'do not delay', 'urgent medical attention', 'seek help immediately',
    'go to a&e', 'visit the er', 'visit er',
]
WITHIN_24H_PHRASES = [
    'as soon as possible', 'asap', 'within 24 hours', 'within a day',
    'see a doctor today', 'visit today', 'today itself',
    'right away', 'without further delay', 'at the earliest',
    'urgently', 'urgent consultation', 'urgent evaluation',
    'should not be ignored', 'must not be ignored', 'needs urgent',
    'requires urgent', 'seek urgent', 'prompt medical',
]
WITHIN_WEEK_PHRASES = [
    'within a week', 'in the next few days', 'next few days',
    'schedule an appointment', 'book an appointment', 'make an appointment',
    'visit a doctor', 'consult a doctor', 'see your doctor', 'see a doctor',
    'get checked', 'get evaluated', 'follow up', 'follow-up',
    'at your earliest convenience', 'soon as convenient',
    'should be checked', 'should be evaluated', 'should be seen',
    'would be advisable', 'i would recommend', 'i recommend consulting',
    'i recommend seeing', 'worth getting checked', 'worth a visit',
    'not urgent but', 'not an emergency but', 'nothing to worry about but',
]
ROUTINE_PHRASES = [
    'routine checkup', 'regular checkup', 'annual checkup',
    'not an emergency', 'non-urgent', 'non urgent',
    'can wait', 'no immediate concern', 'nothing to worry about',
    'monitor at home', 'watch and wait', 'observe at home',
    'self-limiting', 'self limiting', 'will resolve on its own',
]
EXPLICIT_LOW_SEVERITY = [
    'not urgent', 'not serious', 'not an emergency', 'not concerning',
    'nothing to worry', 'nothing to be alarmed', 'no need to worry',
    'non-urgent', 'non urgent', 'minor issue', 'minor problem',
    'mild case', 'common condition',
]
NEGATION_WORDS = ['not ', "n't ", 'no ', 'nothing', 'non']

def _is_negated(text, phrase):
    idx = text.find(phrase)
    if idx == -1: return False
    return any(neg in text[max(0, idx-20):idx] for neg in NEGATION_WORDS)

def extract_urgency(doctor_text):
    """Extract urgency from clinical language with per-phrase negation guard."""
    if not doctor_text: return None
    t = doctor_text.lower()
    for p in IMMEDIATE_PHRASES:
        if p in t and not _is_negated(t, p): return 'immediate'
    for p in WITHIN_24H_PHRASES:
        if p in t and not _is_negated(t, p): return 'within_24_hours'
    for p in WITHIN_WEEK_PHRASES:
        if p in t: return 'within_a_week'
    for p in ROUTINE_PHRASES:
        if p in t: return 'routine'
    return None

SEVERITY_SIGNALS = [
    (['emergency', 'life threatening', 'life-threatening', 'critical',
      'call 911', 'call an ambulance', 'cardiac emergency'], 9),
    (['immediately', 'without delay', 'do not delay', 'do not wait',
      'cannot be ignored', 'must not be ignored',
      'serious condition', 'serious concern', 'serious issue'], 8),
    (['urgently', 'urgent', 'as soon as possible', 'asap',
      'significant concern', 'needs prompt', 'requires prompt',
      'worrying', 'worrisome', 'concerning'], 7),
    (['should see', 'recommend seeing', 'advisable to', 'advised to',
      'get checked', 'get evaluated', 'should not be ignored',
      'should be checked', 'should be seen', 'should be evaluated',
      'i recommend consulting', 'i recommend seeing', 'i would recommend',
      'worth getting checked', 'worth a visit'], 5),
    (['not serious', 'nothing to worry', 'minor', 'mild case',
      'minor issue', 'minor problem', 'not concerning',
      'no need to worry', 'common condition',
      'not urgent', 'not an emergency', 'nothing to be alarmed'], 3),
    (['routine', 'regular checkup', 'normal', 'self-limiting',
      'will resolve', 'resolve on its own', 'monitor at home',
      'non-urgent', 'non urgent'], 2),
]

def extract_severity(doctor_text):
    """Infer severity 1-10. EXPLICIT_LOW_SEVERITY checked first to prevent
    positive phrases winning over explicit negations in the same response."""
    if not doctor_text: return None
    t = doctor_text.lower()
    for p in EXPLICIT_LOW_SEVERITY:
        if p in t: return 3
    for phrases, score in SEVERITY_SIGNALS:
        for p in phrases:
            if p in t and not _is_negated(t, p): return score
    return None


print('Extraction functions loaded. Running sanity checks...')
samples = [
    ('You should see a cardiologist immediately. This could be a cardiac emergency.',
     'Cardiologist', 'immediate', 9),
    ('I recommend consulting a dermatologist. This is not urgent but should be checked.',
     'Dermatologist', 'within_a_week', 3),
    ('Please visit a gastroenterologist as soon as possible for further evaluation.',
     'Gastroenterologist', 'within_24_hours', 7),
    ('This is a routine issue. Monitor at home and see your general physician if it persists.',
     'General Physician', 'routine', 2),
    ('Go to the emergency room immediately. Call an ambulance if needed.',
     None, 'immediate', 9),
]
all_pass = True
for doc_text, exp_spec, exp_urg, exp_sev in samples:
    spec, urg, sev = extract_specialist(doc_text), extract_urgency(doc_text), extract_severity(doc_text)
    ok = spec==exp_spec and urg==exp_urg and sev==exp_sev
    if not ok: all_pass = False
    print(f'  {"PASS" if ok else "FAIL"}  spec={spec} urg={urg} sev={sev} | {doc_text[:55]}')
assert all_pass, 'Sanity checks failed — do not proceed'
print('\nAll 5 checks passed.')

## Cell 3 — Keyword Fallback Functions
Used when extraction returns None (~85% of rows — most HealthCareMagic doctors
write generically without naming specialists).
These functions match on patient symptom text using word groups.

In [ ]:
def _detect_groups(text):
    t = text.lower()
    return {
        'cardiac':     any(w in t for w in ['chest pain','heart attack','palpitation','arrhythmia','angina','cardiac','heart racing','irregular heartbeat']),
        'respiratory': any(w in t for w in ['shortness of breath','difficulty breathing','wheezing','cough','breathless','lung','pneumonia','asthma','tuberculosis']),
        'neuro':       any(w in t for w in ['headache','migraine','seizure','paralysis','stroke','numbness','blurred vision','dizziness','memory','confusion','fainting']),
        'gastro':      any(w in t for w in ['stomach','abdominal','nausea','vomiting','diarrhea','constipation','bloating','heartburn','gastric','bowel','liver','indigestion']),
        'fever':       any(w in t for w in ['fever','chills','sweating','high temperature']),
        'ortho':       any(w in t for w in ['joint pain','back pain','knee','bone','fracture','arthritis','muscle pain','sprain','neck pain','shoulder pain']),
        'skin':        any(w in t for w in ['rash','skin','itch','eczema','psoriasis','bruising','hives','acne']),
        'urinary':     any(w in t for w in ['urination','urine','urinary','kidney','bladder','burning urination','frequent urination','blood in urine']),
        'mental':      any(w in t for w in ['anxiety','depression','panic','stress','mental','mood','sleep','insomnia','suicidal','hallucination']),
        'gynec':       any(w in t for w in ['menstrual','period','vaginal','pregnancy','pelvic','ovary','uterus','discharge']),
        'eye':         any(w in t for w in ['eye pain','vision loss','red eye','blurry vision','eye discharge']),
        'ent':         any(w in t for w in ['ear pain','throat pain','sore throat','nose bleed','hearing loss','tonsil']),
        'diabetes':    any(w in t for w in ['thirst','blood sugar','diabetes','glucose','weight loss']),
    }

def label_specialist_fallback(text):
    g = _detect_groups(text)
    if g['cardiac']:                              return 'Cardiologist'
    if g['respiratory'] and g['fever']:           return 'Pulmonologist'
    if g['fever'] and g['gastro']:                return 'Gastroenterologist'
    if g['fever'] and g['urinary']:               return 'Urologist'
    if g['fever'] and g['neuro']:                 return 'Neurologist'
    if g['respiratory']:                          return 'Pulmonologist'
    if g['neuro']:                                return 'Neurologist'
    if g['gastro']:                               return 'Gastroenterologist'
    if g['ortho']:                                return 'Orthopedist'
    if g['skin']:                                 return 'Dermatologist'
    if g['mental']:                               return 'Psychiatrist'
    if g['gynec']:                                return 'Gynecologist'
    if g['urinary'] or g['diabetes']:             return 'Urologist'
    if g['eye']:                                  return 'Ophthalmologist'
    if g['ent']:                                  return 'ENT Specialist'
    if g['fever']:                                return 'General Physician'
    return 'General Physician'

def label_urgency_fallback(text):
    t = text.lower(); g = _detect_groups(text)
    if any(w in t for w in ['chest pain','heart attack','stroke','not breathing','unconscious','severe bleeding','crushing']): return 'immediate'
    if any(w in t for w in ['high fever','difficulty breathing','vomiting blood','blood in stool','severe pain','confusion','seizure','paralysis','fainting']): return 'within_24_hours'
    if g['cardiac']: return 'within_24_hours'
    if g['fever'] and (g['gastro'] or g['neuro'] or g['urinary']): return 'within_24_hours'
    if any(w in t for w in ['fever','pain','swelling','infection','cough','vomiting','diarrhea','persistent','worsening']): return 'within_a_week'
    return 'routine'

def label_severity_fallback(text):
    t = text.lower(); g = _detect_groups(text)
    base = 4
    if g['cardiac']:                              base = 8
    elif g['respiratory'] and g['fever']:         base = 7
    elif g['respiratory']:                        base = 6
    elif g['neuro'] and g['fever']:               base = 7
    elif g['neuro']:                              base = 6
    elif g['gastro'] and g['fever']:              base = 6
    elif g['fever'] and g['urinary']:             base = 6
    elif g['fever']:                              base = 5
    elif g['gastro']:                             base = 5
    elif g['mental']:                             base = 5
    elif g['ortho']:                              base = 4
    elif g['skin'] or g['eye'] or g['ent']:       base = 3
    if any(w in t for w in ['unbearable','excruciating','worst ever','extreme','crushing']): base = max(base, 9)
    elif any(w in t for w in ['severe','very bad','intense','sharp','high fever','vomiting blood']): base = max(base, 7)
    elif any(w in t for w in ['moderate','significant','persistent','worsening']): base = max(base, 5)
    elif any(w in t for w in ['mild','slight','minor','occasional']): base = min(base, 3)
    if any(w in t for w in ['month','months','chronic','years']): base = min(base+2, 10)
    elif any(w in t for w in ['week','weeks','few days']): base = min(base+1, 10)
    return base

print('Keyword fallback functions loaded.')

## Cell 4 — Hybrid Labeller
Priority: extraction from doctor text first, keyword fallback on patient text second.
Source tracking (`_src_*` fields) is used for the quality report then dropped before training.

In [ ]:
def make_record(patient_text, doctor_text=''):
    patient_text = str(patient_text).strip()[:512]
    doctor_text  = str(doctor_text).strip()
    spec_ext = extract_specialist(doctor_text)
    urg_ext  = extract_urgency(doctor_text)
    sev_ext  = extract_severity(doctor_text)
    return {
        'symptom_text':    patient_text,
        'specialist':      spec_ext if spec_ext is not None else label_specialist_fallback(patient_text),
        'urgency':         urg_ext  if urg_ext  is not None else label_urgency_fallback(patient_text),
        'severity_score':  sev_ext  if sev_ext  is not None else label_severity_fallback(patient_text),
        '_src_specialist': 'extracted' if spec_ext is not None else 'fallback',
        '_src_urgency':    'extracted' if urg_ext  is not None else 'fallback',
        '_src_severity':   'extracted' if sev_ext  is not None else 'fallback',
    }

def print_quality_report(records):
    n = len(records)
    print('\n' + '='*55 + '\nLABEL QUALITY REPORT\n' + '='*55)
    for field, key in [('Specialist','_src_specialist'),('Urgency','_src_urgency'),('Severity','_src_severity')]:
        ext = sum(1 for r in records if r.get(key) == 'extracted')
        pct = ext / n * 100
        print(f'  {field:12} extracted: {ext:6,} ({pct:5.1f}%)  ' + int(pct/5)*'|' + (20-int(pct/5))*'.')
        print(f'               fallback: {n-ext:6,} ({100-pct:5.1f}%)')
    print('='*55)

print('Hybrid labeller loaded.')

## Cell 5 — Load Dataset & Build Labels
Downloads HealthCareMagic-100k (~500MB) and MedQuAD (~100MB) from HuggingFace.
Saves processed CSV to `/kaggle/working/` after first run — loads from cache on restart.

**If this cell prints 'Found cached dataset' — it completes in seconds.**

In [ ]:
PROCESSED_CSV  = os.path.join(DRIVE_DIR, 'combined_dataset_v2.csv')
URGENCY_PKL    = os.path.join(DRIVE_DIR, 'urgency_encoder.pkl')
SPECIALIST_PKL = os.path.join(DRIVE_DIR, 'specialist_encoder.pkl')

if os.path.exists(PROCESSED_CSV):
    print('Found cached dataset — loading...')
    df_raw = pd.read_csv(PROCESSED_CSV)
    with open(URGENCY_PKL,    'rb') as f: urgency_enc    = pickle.load(f)
    with open(SPECIALIST_PKL, 'rb') as f: specialist_enc = pickle.load(f)
    print(f'Loaded {len(df_raw):,} rows')
else:
    records = []
    print('Downloading HealthCareMagic-100k from HuggingFace...')
    hm = load_dataset('lavita/ChatDoctor-HealthCareMagic-100k')['train']
    for item in tqdm(hm, desc='Labelling HealthCareMagic'):
        pt = str(item.get('input',  '')).strip()
        dt = str(item.get('output', '')).strip()
        if len(pt) >= 30: records.append(make_record(pt, dt))

    print('\nDownloading MedQuAD from HuggingFace...')
    try:
        mq = load_dataset('lavita/MedQuAD')['train']
        for item in tqdm(mq, desc='Labelling MedQuAD'):
            text = str(item.get('question', '')).strip()
            if len(text) >= 30: records.append(make_record(text, ''))
    except Exception as e:
        print(f'  MedQuAD unavailable: {e} — continuing without it')

    print_quality_report(records)
    df_raw = pd.DataFrame(records).drop(
        columns=['_src_specialist','_src_urgency','_src_severity'], errors='ignore')

    urgency_enc    = LabelEncoder()
    specialist_enc = LabelEncoder()
    df_raw['urgency_encoded']    = urgency_enc.fit_transform(df_raw['urgency'])
    df_raw['specialist_encoded'] = specialist_enc.fit_transform(df_raw['specialist'])

    df_raw.to_csv(PROCESSED_CSV, index=False)
    with open(URGENCY_PKL,    'wb') as f: pickle.dump(urgency_enc,    f)
    with open(SPECIALIST_PKL, 'wb') as f: pickle.dump(specialist_enc, f)
    print(f'Saved {len(df_raw):,} rows to {DRIVE_DIR}')

print(f'\nRaw dataset   : {len(df_raw):,} rows')
print(f'Specialists   : {list(specialist_enc.classes_)}')
print(f'Urgency       : {list(urgency_enc.classes_)}')

## Cell 5C — Load Disease Dataset from HuggingFace

Loads **`gretelai/symptom_to_diagnosis`** directly from HuggingFace Hub.
No Kaggle account, no CSV upload, no credentials needed — just run the cell.

### Dataset details
| Property | Value |
|----------|-------|
| Source | HuggingFace: `gretelai/symptom_to_diagnosis` |
| Rows | 853 labelled symptom-disease pairs |
| Format | `input_text` (symptom sentence) + `output_text` (disease name) |
| Diseases | ~40 common diseases |
| Access | Completely free, no login required |

### Why this dataset
- Symptom descriptions are already natural language sentences
  (e.g. "I have chest pain, shortness of breath and sweating")
- Plugs directly into our BERT tokenizer with zero preprocessing
- Covers all major disease categories our users report

### Output variables
- `df_disease` — DataFrame with `symptom_text`, `disease`, `disease_encoded`
- `disease_enc` — `LabelEncoder` mapping disease names → integer IDs
- `NUM_DISEASE` — number of unique disease classes (used to size `disease_head`)
- Files cached to `/kaggle/working/` — subsequent runs load instantly from cache


In [ ]:
import pickle
import pandas as pd
from sklearn.preprocessing import LabelEncoder

DISEASE_PKL = os.path.join(DRIVE_DIR, 'disease_encoder.pkl')
DISEASE_CSV = os.path.join(DRIVE_DIR, 'disease_dataset_text.csv')

# ── Option A: Load from cache (instant on 2nd run) ────────────────────────────
if os.path.exists(DISEASE_CSV) and os.path.exists(DISEASE_PKL):
    print('Found cached disease dataset — loading...')
    df_disease  = pd.read_csv(DISEASE_CSV)
    with open(DISEASE_PKL, 'rb') as f: disease_enc = pickle.load(f)
    NUM_DISEASE = len(disease_enc.classes_)
    print(f'Loaded {len(df_disease):,} rows  |  {NUM_DISEASE} diseases')

else:
    # ── Option B: Download from HuggingFace ──────────────────────────────────
    # gretelai/symptom_to_diagnosis — free, no credentials needed.
    # Columns: input_text (symptom sentence), output_text (disease name)
    print('Downloading gretelai/symptom_to_diagnosis from HuggingFace...')
    from datasets import load_dataset as hf_load

    hf_ds = hf_load('gretelai/symptom_to_diagnosis', split='train')
    hf_df = hf_ds.to_pandas()
    print(f'Downloaded: {hf_df.shape}  columns: {list(hf_df.columns)}')

    # ── Identify columns ──────────────────────────────────────────────────────
    # The dataset uses: input_text = symptom, output_text = disease
    # We also handle alternate column names defensively.
    SYMPTOM_COL = None
    DISEASE_COL = None

    # Primary format (gretelai/symptom_to_diagnosis)
    if 'input_text' in hf_df.columns and 'output_text' in hf_df.columns:
        SYMPTOM_COL, DISEASE_COL = 'input_text', 'output_text'
        print('Format detected: input_text / output_text')

    # Alternate format (some HF mirrors)
    elif 'text' in hf_df.columns and 'label' in hf_df.columns:
        SYMPTOM_COL, DISEASE_COL = 'text', 'label'
        print('Format detected: text / label')

    elif 'symptoms' in hf_df.columns and 'diseases' in hf_df.columns:
        SYMPTOM_COL, DISEASE_COL = 'symptoms', 'diseases'
        print('Format detected: symptoms / diseases')

    else:
        # Last resort — take first two columns
        SYMPTOM_COL, DISEASE_COL = hf_df.columns[0], hf_df.columns[1]
        print(f'WARNING: Unknown format. Using columns: {SYMPTOM_COL}, {DISEASE_COL}')

    # ── Build df_disease ──────────────────────────────────────────────────────
    df_disease = pd.DataFrame({
        'symptom_text': hf_df[SYMPTOM_COL].astype(str).str.strip(),
        'disease':      hf_df[DISEASE_COL].astype(str).str.strip(),
    })

    # Drop empty / very short rows
    df_disease = df_disease[df_disease['symptom_text'].str.len() > 10].reset_index(drop=True)
    df_disease = df_disease[df_disease['disease'].str.len() > 2].reset_index(drop=True)

    # ── Encode disease labels ─────────────────────────────────────────────────
    disease_enc = LabelEncoder()
    df_disease['disease_encoded'] = disease_enc.fit_transform(df_disease['disease'])

    # ── Save to cache ─────────────────────────────────────────────────────────
    df_disease.to_csv(DISEASE_CSV, index=False)
    with open(DISEASE_PKL, 'wb') as f: pickle.dump(disease_enc, f)
    print(f'Saved {len(df_disease):,} rows  →  {DISEASE_CSV}')

# ── Final summary ─────────────────────────────────────────────────────────────
NUM_DISEASE = len(disease_enc.classes_)
print(f'\nDisease classes ({NUM_DISEASE} total):')
for i, d in enumerate(disease_enc.classes_):
    print(f'  {i:>3}. {d}')

print(f'\nSample rows:')
for _, row in df_disease.sample(3, random_state=42).iterrows():
    print(f'  SYMPTOM : {str(row["symptom_text"])[:80]}')
    print(f'  DISEASE : {row["disease"]}')
    print()


## Cell 5D — Auto-Label Risk Factors from HealthCareMagic

Generates **weak multi-label risk factor annotations** from the existing
HealthCareMagic text data using regex pattern matching.

### Why "weak labels"?
We have no dataset with `(symptom text, risk_factors[])` pairs. Instead, we
extract risk signals directly from the patient's own words using carefully
crafted patterns. This is standard clinical NLP practice — it's called
**distant supervision**.

### 10 risk factor categories
| ID | Risk Factor | Example trigger words |
|----|-------------|----------------------|
| 0 | hypertension | "blood pressure", "bp high", "hypertension" |
| 1 | diabetes | "diabetic", "blood sugar", "insulin" |
| 2 | smoking | "smoker", "cigarette", "tobacco" |
| 3 | age_above_40 | "45 year old", "age: 52" |
| 4 | obesity | "obese", "overweight", "bmi 32" |
| 5 | family_history | "father had", "family history", "hereditary" |
| 6 | stress | "stress", "anxiety", "work pressure" |
| 7 | alcohol | "alcohol", "drinking", "alcoholic" |
| 8 | sedentary | "sedentary", "no exercise", "inactive" |
| 9 | poor_diet | "junk food", "high fat", "unhealthy diet" |

### Output
- `RISK_FACTOR_LABELS` — list of 10 risk factor names (ordered)
- `df_risk` — HealthCareMagic rows with a `risk_vector` column (10-dim binary list)
- Each row can have 0–10 risk factors active simultaneously (multi-label)


In [ ]:
import re
import numpy as np

# ── Risk factor definitions ──────────────────────────────────────────────────
# Each entry: (label_name, compiled_regex_pattern)
# Patterns are designed to be high-precision (few false positives) even if
# they miss some cases — for weak labels, precision > recall.

RISK_FACTOR_LABELS = [
    'hypertension',
    'diabetes',
    'smoking_history',
    'age_above_40',
    'obesity',
    'family_history',
    'stress',
    'alcohol',
    'sedentary_lifestyle',
    'poor_diet',
]

RISK_PATTERNS = {
    'hypertension':       re.compile(
        r'\b(hypertension|high blood pressure|bp[:\s]+high|elevated bp|'
        r'blood pressure[:\s]+\d{3}|bp[:\s]+\d{3}|stage [12] hypertension)\b', re.I),
    'diabetes':           re.compile(
        r'\b(diabetic|diabetes|blood sugar|insulin|glucose|hba1c|'
        r'fasting sugar|post prandial|type [12] diabetes|t2dm|t1dm)\b', re.I),
    'smoking_history':    re.compile(
        r'\b(smoker|smoking|cigarette|tobacco|smokes?|nicotine|'
        r'pack year|chain smok|ex.smok)\b', re.I),
    'age_above_40':       re.compile(
        r'\b(age[:\s]+[4-9]\d|[4-9]\d[\s-]?year[\s-]?old|'
        r'aged? [4-9]\d|[4-9]\d year old male|[4-9]\d year old female|'
        r'\b[4-9]\dM\b|\b[4-9]\dF\b)\b', re.I),
    'obesity':            re.compile(
        r'\b(obese|obesity|overweight|bmi[:\s]+[3-9]\d|'
        r'body mass index[:\s]+[3-9]\d|morbidly obese|weight gain)\b', re.I),
    'family_history':     re.compile(
        r'\b(family history|father had|mother had|hereditary|'
        r'runs in (the )?family|genetic predisposition|familial|'
        r'sibling(s)? (had|have|with))\b', re.I),
    'stress':             re.compile(
        r'\b(stress(ed|ful)?|anxiety|anxious|work pressure|burnout|'
        r'mental pressure|emotional stress|overwhelmed|tension)\b', re.I),
    'alcohol':            re.compile(
        r'\b(alcohol(ic)?|drinking|drink(s)? alcohol|beer|wine|whiskey|'
        r'liquor|substance abuse|binge drink)\b', re.I),
    'sedentary_lifestyle':re.compile(
        r'\b(sedentary|no exercise|inactive|physically inactive|'
        r'desk job|sitting (all day|for long)|lack of physical activity|'
        r'does not exercise|rarely exercises)\b', re.I),
    'poor_diet':          re.compile(
        r'\b(junk food|fast food|high fat|high sugar|processed food|'
        r'unhealthy diet|poor diet|skips meals|irregular meals|high sodium)\b', re.I),
}

def extract_risk_factors(text):
    """
    Extract risk factors from free text using regex patterns.
    Returns a binary list of length 10 — one flag per risk factor.
    Multi-label: multiple factors can be active simultaneously.
    """
    text = str(text)
    return [1 if RISK_PATTERNS[label].search(text) else 0
            for label in RISK_FACTOR_LABELS]

# ── Apply to HealthCareMagic data ────────────────────────────────────────────
print('Extracting risk factors from HealthCareMagic data...')
risk_vectors = [extract_risk_factors(row['symptom_text']) for _, row in df_raw.iterrows()]
df_risk = df_raw[['symptom_text']].copy()
df_risk['risk_vector'] = risk_vectors

# ── Quality report ────────────────────────────────────────────────────────────
print('\n' + '='*55)
print('RISK FACTOR COVERAGE REPORT')
print('='*55)
total = len(df_risk)
risk_array = np.array(risk_vectors)
for i, label in enumerate(RISK_FACTOR_LABELS):
    count = risk_array[:, i].sum()
    pct   = count / total * 100
    bar   = int(pct / 2) * '|' + (25 - int(pct / 2)) * '.'
    print(f'  {label:<22} {count:>6,}  ({pct:5.1f}%)  {bar}')

# Rows with at least one risk factor
has_risk = (risk_array.sum(axis=1) > 0).sum()
print(f'\n  Rows with ≥1 risk factor : {has_risk:,} / {total:,} ({has_risk/total*100:.1f}%)')
print(f'  Rows with 0 risk factors  : {total - has_risk:,} (no signal in text)')
print(f'  Avg risk factors per row  : {risk_array.sum(axis=1).mean():.2f}')
print('='*55)

# ── Sample output ─────────────────────────────────────────────────────────────
print('\nSample extractions:')
sample_texts = [
    '55M | chest pain, smoker for 20 years, father had heart attack, BP 160/100',
    '35F | anxiety, work pressure, sedentary lifestyle, junk food daily',
    '22M | fever and cough for 3 days',
]
for t in sample_texts:
    vec = extract_risk_factors(t)
    detected = [RISK_FACTOR_LABELS[i] for i, v in enumerate(vec) if v == 1]
    print(f'  TEXT     : {t[:60]}')
    print(f'  DETECTED : {detected if detected else ["none"]}')
    print()


## Cell 5E — Disease–Risk Factor Knowledge Map (Auto-generated)

Builds a `DISEASE_RISK_MAP` that covers **every disease in the actual dataset**,
not just a hardcoded list of 41. This cell dynamically reads `disease_enc.classes_`
and creates entries for all classes.

### Two-tier approach
1. **Handcrafted entries** — precise risk factors for ~50 common diseases
   (based on WHO/Mayo Clinic references)
2. **Auto-generated entries** — all remaining diseases get a sensible generic
   set of risk factors derived from their disease category (cardiac, respiratory,
   neurological, etc.) detected from the disease name itself

This means `DISEASE_RISK_MAP` always has 100% coverage regardless of which
HuggingFace dataset version loads — no more `WARNING: 385 diseases missing` errors.

### How linked risks are used at inference
After BERT predicts top-3 diseases, we look up each disease in this map and
surface its risk factors to the user — adding clinical depth even when the
symptom text itself doesn't mention them explicitly.


In [ ]:
# ── Handcrafted risk map for common diseases ─────────────────────────────────
# These are clinically precise entries for the most important diseases.
# Keys are matched case-insensitively against disease_enc.classes_.

HANDCRAFTED_RISK_MAP = {
    # Cardiovascular
    'heart attack':                ['Hypertension', 'Diabetes', 'Smoking', 'High cholesterol',
                                    'Obesity', 'Family cardiac history', 'Sedentary lifestyle', 'Age above 45'],
    'myocardial infarction':       ['Hypertension', 'Diabetes', 'Smoking', 'High cholesterol',
                                    'Obesity', 'Family cardiac history', 'Age above 45'],
    'hypertension':                ['Obesity', 'High sodium diet', 'Sedentary lifestyle', 'Stress',
                                    'Alcohol', 'Age above 40', 'Family history', 'Diabetes'],
    'hypertensive heart disease':  ['Hypertension', 'Age above 50', 'Obesity', 'Diabetes', 'Smoking'],
    'coronary atherosclerosis':    ['Hypertension', 'High cholesterol', 'Diabetes', 'Smoking', 'Obesity'],
    'atrial fibrillation':         ['Age above 60', 'Hypertension', 'Heart disease', 'Alcohol', 'Obesity'],
    'stroke':                      ['Hypertension', 'Diabetes', 'Smoking', 'Atrial fibrillation',
                                    'High cholesterol', 'Age above 55', 'Obesity'],
    'transient ischemic attack':   ['Hypertension', 'Diabetes', 'Smoking', 'High cholesterol', 'Age above 55'],
    'varicose veins':              ['Prolonged standing', 'Obesity', 'Pregnancy', 'Age above 50', 'Family history'],
    'pericarditis':                ['Viral infection', 'Autoimmune disease', 'Heart surgery', 'Kidney failure'],
    'heart block':                 ['Age above 60', 'Heart disease', 'Medications', 'Lyme disease'],
    'mitral valve disease':        ['Rheumatic fever', 'Age above 50', 'Mitral valve prolapse'],

    # Metabolic / Endocrine
    'diabetes':                    ['Obesity', 'Sedentary lifestyle', 'Age above 45', 'Family history',
                                    'Poor diet', 'Hypertension', 'High cholesterol'],
    'type 1 diabetes':             ['Family history', 'Autoimmune conditions', 'Viral infections'],
    'type 2 diabetes':             ['Obesity', 'Sedentary lifestyle', 'Age above 45', 'Family history',
                                    'Poor diet', 'Hypertension'],
    'gestational diabetes':        ['Obesity', 'Previous gestational diabetes', 'Family history',
                                    'Age above 25', 'Polycystic ovary syndrome'],
    'hyperthyroidism':             ['Family history', 'Female gender', 'Autoimmune conditions', 'Iodine excess'],
    'hypothyroidism':              ['Family history', 'Autoimmune conditions', 'Female gender',
                                    'Age above 60', 'Iodine deficiency'],
    'hashimoto thyroiditis':       ['Female gender', 'Family history', 'Autoimmune conditions', 'Age 30-50'],
    'obesity':                     ['Poor diet', 'Sedentary lifestyle', 'Genetics', 'Stress',
                                    'Hormonal imbalance', 'Sleep deprivation'],
    'hypoglycemia':                ['Diabetes medications', 'Fasting', 'Alcohol', 'Liver disease'],
    'hypercholesterolemia':        ['Poor diet', 'Sedentary lifestyle', 'Obesity', 'Family history',
                                    'Diabetes', 'Hypothyroidism'],
    'gout':                        ['High purine diet', 'Alcohol', 'Obesity', 'Kidney disease',
                                    'Diuretic use', 'Male gender'],

    # Respiratory
    'pneumonia':                   ['Age extremes', 'Smoking', 'Weak immune system',
                                    'Chronic lung disease', 'Alcohol abuse', 'Hospitalization'],
    'tuberculosis':                ['HIV/AIDS', 'Malnutrition', 'Smoking', 'Diabetes',
                                    'Overcrowding', 'Weak immunity', 'Alcohol abuse'],
    'bronchial asthma':            ['Family history', 'Allergies', 'Smoking', 'Air pollution',
                                    'Obesity', 'Respiratory infections in childhood'],
    'asthma':                      ['Family history', 'Allergies', 'Smoking exposure', 'Air pollution',
                                    'Obesity', 'Childhood respiratory infections'],
    'emphysema':                   ['Smoking', 'Age above 40', 'Air pollution', 'Alpha-1 antitrypsin deficiency'],
    'bronchitis':                  ['Smoking', 'Air pollution', 'Weak immunity', 'Cold weather exposure'],
    'pulmonary fibrosis':          ['Smoking', 'Age above 50', 'Male gender', 'Environmental dust exposure'],
    'pulmonary congestion':        ['Heart failure', 'Hypertension', 'Kidney disease', 'Fluid overload'],
    'cystic fibrosis':             ['Genetic (CFTR mutation)', 'Family history'],
    'influenza (flu)':             ['Age extremes', 'Weak immunity', 'No vaccination',
                                    'Crowded environments', 'Healthcare workers'],

    # Gastrointestinal
    'gastroesophageal reflux disease (gerd)': ['Obesity', 'Smoking', 'Alcohol', 'High fat diet',
                                               'Pregnancy', 'Stress', 'Hiatal hernia'],
    'peptic ulcer disease':        ['H. pylori infection', 'NSAID use', 'Smoking', 'Stress', 'Alcohol'],
    'gastroenteritis (stomach flu)': ['Contaminated food/water', 'Poor hygiene', 'Weak immunity',
                                      'Travel to high-risk areas'],
    'gastroenteritis':             ['Contaminated food/water', 'Poor hygiene', 'Weak immunity'],
    'indigestion':                 ['Stress', 'Fatty foods', 'Eating too fast', 'H. pylori',
                                    'Smoking', 'Alcohol', 'Certain medications'],
    'cirrhosis':                   ['Chronic alcohol use', 'Hepatitis B/C', 'Obesity',
                                    'Non-alcoholic fatty liver', 'Autoimmune hepatitis'],
    'hepatitis due to a toxin':    ['Alcohol abuse', 'Certain medications', 'Herbal supplements',
                                    'Industrial chemicals'],
    'nonalcoholic liver disease (nash)': ['Obesity', 'Type 2 diabetes', 'High cholesterol',
                                          'Metabolic syndrome', 'Sedentary lifestyle'],
    'gastrointestinal hemorrhage': ['Peptic ulcer', 'NSAIDs', 'Alcohol', 'Liver disease',
                                    'Anticoagulant use'],
    'hemorrhoids':                 ['Chronic constipation', 'Low-fibre diet', 'Sedentary lifestyle',
                                    'Obesity', 'Pregnancy', 'Straining during bowel movements'],
    'colonic polyp':               ['Age above 50', 'Family history', 'High-fat diet',
                                    'Sedentary lifestyle', 'Smoking', 'Alcohol'],
    'diverticulosis':              ['Age above 40', 'Low-fibre diet', 'Sedentary lifestyle',
                                    'Obesity', 'Smoking'],

    # Neurological
    'migraine':                    ['Family history', 'Female gender', 'Stress', 'Hormonal changes',
                                    'Sleep disruption', 'Certain foods/drinks', 'Weather changes'],
    'chronic migraine':            ['Family history', 'Female gender', 'Stress', 'Medication overuse',
                                    'Sleep disorders', 'Hormonal changes'],
    'tension headache':            ['Stress', 'Poor posture', 'Eye strain', 'Dehydration', 'Lack of sleep'],
    'chronic headache':            ['Stress', 'Medication overuse', 'Sleep disorders', 'Anxiety', 'Depression'],
    'parkinson disease':           ['Age above 60', 'Male gender', 'Family history', 'Pesticide exposure',
                                    'Head injury history'],
    'epilepsy':                    ['Family history', 'Head injury', 'Stroke', 'Brain infections',
                                    'Prenatal brain damage'],
    'multiple sclerosis':          ['Female gender', 'Age 20-40', 'Family history', 'Vitamin D deficiency',
                                    'Smoking', 'Certain viral infections'],
    'encephalitis':                ['Viral infection', 'Weak immunity', 'Mosquito/tick exposure',
                                    'No vaccination'],
    'concussion':                  ['Contact sports', 'Previous head injury', 'Age extremes',
                                    'Lack of protective gear'],
    'intracranial hemorrhage':     ['Hypertension', 'Anticoagulant use', 'Head trauma',
                                    'Age above 55', 'Aneurysm'],

    # Infectious
    'malaria':                     ['Travel to endemic regions', 'No prophylaxis',
                                    'Outdoor activity at dusk/dawn', 'Weak immunity'],
    'typhoid':                     ['Contaminated food/water', 'Travel to endemic areas',
                                    'Poor sanitation', 'No vaccination'],
    'dengue fever':                ['Tropical climate', 'Aedes mosquito exposure',
                                    'Urban living', 'Previous dengue infection'],
    'dengue':                      ['Tropical climate', 'Aedes mosquito exposure', 'Urban living'],
    'mononucleosis':               ['Young adults (15-25)', 'Close contact (kissing)',
                                    'Weak immunity', 'Crowded living'],
    'rabies':                      ['Animal bites', 'No post-exposure prophylaxis',
                                    'Rural living', 'Animal handlers'],
    'west nile virus':             ['Mosquito exposure', 'Age above 50', 'Weak immunity',
                                    'Summer/autumn season'],
    'leishmaniasis':               ['Sandfly exposure', 'Travel to endemic regions',
                                    'Weak immunity', 'Malnutrition'],
    'chagas disease':              ['Travel to Latin America', 'Triatomine bug exposure',
                                    'Blood transfusion', 'Weak immunity'],

    # Skin
    'psoriasis':                   ['Family history', 'Stress', 'Infections', 'Smoking',
                                    'Alcohol', 'Obesity', 'Certain medications'],
    'eczema':                       ['Family history of atopy', 'Allergen exposure', 'Stress',
                                    'Dry skin', 'Irritant contact'],
    'contact dermatitis':          ['Allergen/irritant exposure', 'Atopic dermatitis history',
                                    'Occupational chemical exposure'],
    'fungal infection of the skin':['Diabetes', 'Weak immunity', 'Excessive sweating',
                                    'Tight clothing', 'Antibiotic use'],
    'acne':                        ['Hormonal changes', 'Family history', 'High-glycemic diet',
                                    'Stress', 'Certain medications'],
    'urticaria (hives)':           ['Allergen exposure', 'Stress', 'Infections', 'Medications',
                                    'Temperature extremes'],
    'scabies':                     ['Close personal contact', 'Crowded living',
                                    'Weak immunity', 'Sexual contact with infected person'],
    'cellulitis':                  ['Skin injury', 'Diabetes', 'Obesity', 'Weak immunity',
                                    'Poor circulation', 'Lymphedema'],

    # Musculoskeletal
    'arthritis':                   ['Age above 50', 'Family history', 'Obesity',
                                    'Previous joint injury', 'Female gender', 'Smoking'],
    'rheumatoid arthritis':        ['Female gender', 'Age 40-60', 'Family history',
                                    'Smoking', 'Hormonal factors'],
    'osteoarthritis':              ['Age above 50', 'Obesity', 'Previous joint injury',
                                    'Sedentary lifestyle', 'Female gender'],
    'gout':                        ['High purine diet', 'Alcohol', 'Obesity',
                                    'Kidney disease', 'Diuretics', 'Male gender'],
    'osteoporosis':                ['Female gender', 'Age above 50', 'Low calcium/vitamin D',
                                    'Sedentary lifestyle', 'Smoking', 'Alcohol'],
    'fibromyalgia':                ['Female gender', 'Stress', 'Sleep disorders',
                                    'Anxiety/depression', 'Family history', 'Trauma'],
    'chronic knee pain':           ['Obesity', 'Previous knee injury', 'Age above 40',
                                    'Sedentary lifestyle', 'Repetitive knee stress'],
    'spondylosis':                 ['Age above 40', 'Poor posture', 'Sedentary lifestyle',
                                    'Neck/back strain', 'Family history'],
    'spondylitis':                 ['Male gender', 'Age 17-45', 'Family history (HLA-B27)',
                                    'Inflammatory bowel disease'],
    'sciatica':                    ['Age 30-50', 'Obesity', 'Sedentary lifestyle',
                                    'Prolonged sitting', 'Heavy lifting'],
    'plantar fasciitis':           ['Obesity', 'Prolonged standing', 'Poor footwear',
                                    'Tight calf muscles', 'Age 40-60'],

    # Urological / Renal
    'urinary tract infection (uti)': ['Female gender', 'Sexual activity', 'Diabetes',
                                      'Menopause', 'Urinary catheters', 'Kidney stones'],
    'urinary tract infection':     ['Female gender', 'Sexual activity', 'Diabetes',
                                    'Menopause', 'Urinary catheters'],
    'kidney stone':                ['Low fluid intake', 'High protein/sodium diet', 'Obesity',
                                    'Family history', 'Hyperparathyroidism'],
    'urinary stones (kidney stones)': ['Low fluid intake', 'High protein diet', 'Obesity',
                                       'Family history', 'Recurrent UTIs'],
    'chronic kidney disease':      ['Diabetes', 'Hypertension', 'Age above 60',
                                    'Family history', 'Obesity', 'Smoking'],
    'kidney failure':              ['Diabetes', 'Hypertension', 'Chronic kidney disease',
                                    'NSAID overuse', 'Contrast dye exposure'],
    'pyelonephritis':              ['UTI history', 'Kidney stones', 'Urinary catheters',
                                    'Diabetes', 'Female gender'],
    'polycystic kidney disease':   ['Family history (autosomal dominant)', 'Hypertension'],
    'diabetic kidney disease':     ['Poorly controlled diabetes', 'Hypertension',
                                    'Long diabetes duration', 'Smoking'],

    # Gynaecological / Reproductive
    'polycystic ovarian syndrome': ['Obesity', 'Family history', 'Insulin resistance',
                                    'Sedentary lifestyle', 'High androgen levels'],
    'endometriosis':               ['Family history', 'Early menstruation', 'Short menstrual cycles',
                                    'Never given birth', 'Reproductive tract abnormalities'],
    'uterine fibroids':            ['Age 30-50', 'African descent', 'Obesity', 'Early menstruation',
                                    'Family history', 'No pregnancies'],
    'ectopic pregnancy':           ['Previous ectopic pregnancy', 'Pelvic inflammatory disease',
                                    'Fallopian tube surgery', 'IUD use', 'Smoking'],
    'vaginal yeast infection':     ['Antibiotic use', 'Diabetes', 'Pregnancy', 'Weak immunity',
                                    'Hormonal contraceptives', 'High-sugar diet'],
    'preeclampsia':                ['First pregnancy', 'Multiple pregnancy', 'Obesity',
                                    'Hypertension', 'Diabetes', 'Age above 35'],
    'gestational hypertension':    ['First pregnancy', 'Obesity', 'Multiple pregnancy',
                                    'Age above 35', 'Family history'],
    'postpartum depression':       ['History of depression', 'Stressful life events',
                                    'Lack of support', 'Difficult pregnancy', 'Hormonal changes'],

    # Mental health
    'anxiety':                     ['Stress', 'Trauma history', 'Family history',
                                    'Chronic illness', 'Substance use', 'Major life changes'],
    'depression':                  ['Family history', 'Stress', 'Trauma', 'Chronic illness',
                                    'Substance abuse', 'Isolation', 'Hormonal changes'],
    'panic disorder':              ['Family history', 'Stress', 'Major life changes',
                                    'Substance use', 'Anxiety disorders'],
    'panic attack':                ['Stress', 'Anxiety disorder', 'Stimulant use',
                                    'Thyroid disorders', 'Family history'],
    'bipolar disorder':            ['Family history', 'Stress', 'Substance abuse',
                                    'Sleep disruption', 'Major life events'],
    'attention deficit hyperactivity disorder (adhd)': ['Family history', 'Premature birth',
                                    'Low birth weight', 'Prenatal toxin exposure'],
    'primary insomnia':            ['Stress', 'Anxiety/depression', 'Poor sleep hygiene',
                                    'Shift work', 'Caffeine', 'Chronic pain'],
    'anorexia nervosa':            ['Female gender', 'Age 13-18', 'Family history of eating disorders',
                                    'Perfectionism', 'Low self-esteem', 'Societal pressure'],

    # Haematological
    'anemia':                      ['Iron/B12/folate deficiency', 'Chronic disease', 'Blood loss',
                                    'Genetic conditions', 'Pregnancy', 'Poor diet'],
    'leukemia':                    ['Previous chemotherapy', 'Radiation exposure', 'Family history',
                                    'Genetic syndromes', 'Smoking', 'Chemical exposure'],
    'lymphoma':                    ['Weak immunity', 'Certain viral infections (EBV)', 'Family history',
                                    'Age above 60 or young adults'],
    'hemophilia':                  ['X-linked genetic inheritance', 'Family history', 'Male gender'],
    'thrombocytopenia':            ['Autoimmune disease', 'Medications', 'Viral infections',
                                    'Chemotherapy', 'Alcohol abuse'],

    # Eye
    'diabetic retinopathy':        ['Poorly controlled diabetes', 'Hypertension',
                                    'Long diabetes duration', 'Smoking'],
    'macular degeneration':        ['Age above 55', 'Smoking', 'Family history',
                                    'Hypertension', 'Obesity', 'High-fat diet'],
    'glaucoma':                    ['Age above 60', 'Family history', 'Diabetes',
                                    'Hypertension', 'Eye injury', 'Corticosteroid use'],
    'open-angle glaucoma':         ['Age above 60', 'Family history', 'Diabetes',
                                    'Hypertension', 'African descent'],
    'cataracts':                   ['Age above 60', 'Smoking', 'Diabetes', 'UV exposure',
                                    'Corticosteroid use', 'Eye injury'],
    'retinal detachment':          ['High myopia', 'Age above 50', 'Previous eye surgery',
                                    'Eye trauma', 'Family history'],

    # ENT
    'chronic sinusitis':           ['Allergies', 'Nasal polyps', 'Deviated septum',
                                    'Weak immunity', 'Smoking', 'Air pollution'],
    'otitis media':                ['Young children', 'Day care attendance', 'No breastfeeding',
                                    'Smoking exposure', 'Allergies'],
    'meniere disease':             ['Autoimmune conditions', 'Viral infections', 'Genetic factors',
                                    'Age 40-60', 'High sodium diet'],

    # Drug/substance
    'drug reaction':               ['Multiple medications', 'Previous drug allergies',
                                    'Autoimmune conditions', 'Renal/hepatic impairment'],
    'alcohol use disorder':        ['Family history', 'Mental health disorders', 'Early alcohol use',
                                    'Stress', 'Social environment'],
    'opioid use disorder':         ['Prescription opioid use', 'Family history of addiction',
                                    'Mental health disorders', 'Chronic pain', 'Social factors'],
}

# ── Auto-generate risk entries for any disease not in handcrafted map ─────────
# We detect the category from keywords in the disease name and assign
# sensible generic risk factors. This guarantees 100% coverage.

CATEGORY_RISKS = {
    'cancer':       ['Smoking', 'Age above 50', 'Family history', 'Environmental toxin exposure',
                     'Chronic inflammation', 'Obesity', 'Alcohol use'],
    'glaucoma':     ['Age above 60', 'Family history', 'Diabetes', 'Hypertension', 'Corticosteroid use'],
    'fracture':     ['Osteoporosis', 'Age above 60', 'Trauma', 'Calcium/vitamin D deficiency',
                     'Corticosteroid use'],
    'anemia':       ['Iron/B12/folate deficiency', 'Chronic disease', 'Blood loss', 'Poor diet'],
    'poisoning':    ['Accidental ingestion', 'Substance abuse', 'Occupational exposure',
                     'Medication errors'],
    'disorder':     ['Genetic predisposition', 'Family history', 'Chronic stress',
                     'Environmental factors'],
    'infection':    ['Weak immune system', 'Close contact with infected person',
                     'Poor hygiene', 'No vaccination'],
    'syndrome':     ['Genetic factors', 'Family history', 'Environmental triggers'],
    'injury':       ['Physical trauma', 'Sports activities', 'Occupational hazard',
                     'Lack of protective equipment'],
    'stone':        ['Low fluid intake', 'High mineral diet', 'Family history',
                     'Recurrent infections'],
    'deficiency':   ['Poor diet', 'Malabsorption', 'Inadequate sun exposure',
                     'Vegetarian/vegan diet', 'Chronic disease'],
    'hypertension': ['Obesity', 'High sodium diet', 'Stress', 'Family history', 'Sedentary lifestyle'],
    'diabetes':     ['Obesity', 'Sedentary lifestyle', 'Family history', 'Poor diet'],
    'arthritis':    ['Age above 50', 'Obesity', 'Joint injury history', 'Family history'],
    'pregnancy':    ['Age above 35', 'Multiple pregnancies', 'Obesity',
                     'Pre-existing medical conditions'],
    'hernia':       ['Heavy lifting', 'Obesity', 'Chronic cough', 'Constipation', 'Age above 40'],
    'incontinence': ['Female gender', 'Childbirth', 'Age above 50', 'Obesity', 'Chronic cough'],
    'hepatitis':    ['Viral infection', 'Blood transfusion', 'Alcohol use', 'Intravenous drug use'],
    'dermatitis':   ['Allergen exposure', 'Skin irritants', 'Family history of atopy', 'Stress'],
    'neuropathy':   ['Diabetes', 'Vitamin deficiency', 'Alcohol', 'Kidney disease',
                     'Chemotherapy'],
    'default':      ['Genetic predisposition', 'Age-related changes', 'Lifestyle factors',
                     'Environmental exposure', 'Family history'],
}

def _auto_risks(disease_name):
    # Return risk factors for a disease not in handcrafted map using category matching.
    dn = disease_name.lower()
    for keyword, risks in CATEGORY_RISKS.items():
        if keyword in dn:
            return risks
    return CATEGORY_RISKS['default']

# ── Build the final DISEASE_RISK_MAP covering ALL encoder classes ─────────────
DISEASE_RISK_MAP = {}
auto_count = 0
for disease in disease_enc.classes_:
    key = disease.lower()
    # Check handcrafted map (case-insensitive)
    matched = None
    for hk, hv in HANDCRAFTED_RISK_MAP.items():
        if hk == key or hk in key or key in hk:
            matched = hv
            break
    if matched:
        DISEASE_RISK_MAP[disease] = matched
    else:
        DISEASE_RISK_MAP[disease] = _auto_risks(disease)
        auto_count += 1

handcrafted_count = len(DISEASE_RISK_MAP) - auto_count
print(f'DISEASE_RISK_MAP built:')
print(f'  Total diseases    : {len(DISEASE_RISK_MAP)}')
print(f'  Handcrafted       : {handcrafted_count}  (clinically precise)')
print(f'  Auto-generated    : {auto_count}  (category-based fallback)')
print(f'  Coverage          : 100%  (no missing diseases)')

def get_linked_risks(top_disease_names):
    """
    Given a list of predicted disease names, return their
    aggregated known clinical risk factors (deduplicated).
    """
    risks, seen = [], set()
    for disease in top_disease_names:
        for risk in DISEASE_RISK_MAP.get(disease, CATEGORY_RISKS['default']):
            if risk.lower() not in seen:
                risks.append(risk)
                seen.add(risk.lower())
    return risks

# ── Quick test ─────────────────────────────────────────────────────────────────
print('\nSample linked risks:')
sample_diseases = list(disease_enc.classes_)[:3]
for disease in sample_diseases:
    risks = get_linked_risks([disease])
    print(f'  {disease:<45} -> {risks[:3]}')


## Cell 5B — Balance Dataset to 3,000 Rows Per Specialist Class

### Why 3,000 instead of 5,000
Previous run with 5,000/class: Stage 3 epoch 1 hit spec_acc=0.80 immediately.
This shows the model learns fast once BERT unfreezes — 3,000 rows per class
provides sufficient signal with ~33% less training time per epoch.

### Balancing strategy
- **Majority classes** (>3000 rows): undersample — random sample of exactly 3,000
- **Minority classes** (<3000 rows): oversample with replacement — repeat rows to reach 3,000
- Result: 12 × 3,000 = 36,000 rows, 1.0x imbalance

### Known trade-off
Ophthalmologist (189 original rows) is repeated ~15x. The model memorises those
specific examples rather than generalising. Acceptable — no more data exists.

### Training time impact
36,000 rows / batch 16 = 2,250 batches/epoch → ~8 min/epoch (Stage 1/2)
Stage 3 with BERT unfrozen → ~22 min/epoch → 10 epochs = ~3.5 hrs

In [ ]:
ROWS_PER_CLASS = 3000
RANDOM_SEED    = 42

balanced_parts = []
for specialist in specialist_enc.classes_:
    subset = df_raw[df_raw['specialist'] == specialist]
    n      = len(subset)
    if n >= ROWS_PER_CLASS:
        sampled = subset.sample(n=ROWS_PER_CLASS, random_state=RANDOM_SEED)
        action  = 'undersampled'
    else:
        sampled = subset.sample(n=ROWS_PER_CLASS, replace=True, random_state=RANDOM_SEED)
        action  = f'oversampled (x{ROWS_PER_CLASS//max(n,1):.0f})'
    balanced_parts.append(sampled)
    print(f'  {specialist:<25} {n:>6,} -> {ROWS_PER_CLASS:,}  [{action}]')

df = pd.concat(balanced_parts).sample(
    frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

df['urgency_encoded']    = urgency_enc.transform(df['urgency'])
df['specialist_encoded'] = specialist_enc.transform(df['specialist'])

NUM_URGENCY    = len(urgency_enc.classes_)
NUM_SPECIALIST = len(specialist_enc.classes_)

train_df, val_df = train_test_split(
    df, test_size=0.15, random_state=RANDOM_SEED, stratify=df['specialist'])

print(f'\nBalanced: {len(df):,} rows  (was {len(df_raw):,})')
print(f'Train: {len(train_df):,}  Val: {len(val_df):,}')
print(f'Batches/epoch: {int(len(train_df)/16):,}')
print(f'Est Stage 1/2 time/epoch: ~{int(len(train_df)/16*0.26/60)} min')
print(f'Est Stage 3 time/epoch:   ~{int(len(train_df)/16*0.41/60)} min')

print('\nUrgency distribution after balancing:')
urg_new = df['urgency'].value_counts()
for name, count in urg_new.items():
    pct = count / len(df) * 100
    print(f'  {name:<20} {count:>5,}  ({pct:5.1f}%)')
print(f'  Urgency ratio: {urg_new.iloc[0]/urg_new.iloc[-1]:.1f}x  (weighted loss handles this)')

## Cell 6 — PyTorch Dataset
Tokenises symptom text and returns label tensors.
Severity is normalised to 0-1 range (from 1-10) for stable MSE loss —
multiply by 9 and add 1 at inference time to get back to the 1-10 display scale.

In [ ]:
class MedDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tokenizer(str(row['symptom_text']),
            max_length=self.max_len, padding='max_length',
            truncation=True, return_tensors='pt')
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'severity_cls':   torch.tensor(int(row['severity_score']) - 1,          dtype=torch.long),
            'severity_reg':   torch.tensor((float(row['severity_score']) - 1) / 9,  dtype=torch.float),
            'urgency':        torch.tensor(int(row['urgency_encoded']),              dtype=torch.long),
            'specialist':     torch.tensor(int(row['specialist_encoded']),           dtype=torch.long),
        }
print('MedDataset defined.')

## Cell 6B — Extended MedDataset with Disease + Risk Factor Labels

Extends the `MedDataset` class to also return two new label types:
- **`disease`** — integer class ID (from HuggingFace symptom-to-diagnosis data)
- **`risk_vector`** — 10-dim binary tensor for multi-label risk factor classification

### Two dataset modes
1. **`has_disease=True`** — used when training the disease head (HuggingFace data)
2. **`has_risk=True`** — used when training the risk factor head (HCM auto-labeled data)
3. Both can be False — original behaviour for specialist/urgency/severity training

### Risk vector format
Each position in the 10-dim vector maps to a risk factor (defined in Cell 5D).
A value of `1.0` means that risk factor was detected via regex in the patient's text.
`BCEWithLogitsLoss` handles this multi-label setup — each factor is an independent
binary decision, unlike CrossEntropyLoss which forces a single winner.

### Note on `stratify` parameter
With very small per-class counts (the HuggingFace dataset has ~2 rows/disease in some classes),
`stratify` may fail. The code handles this gracefully with a fallback to non-stratified split.


In [ ]:
class MedDatasetExtended(Dataset):
    """
    Extended dataset supporting disease classification and multi-label
    risk factor targets on top of the existing specialist/urgency/severity labels.

    Parameters
    ----------
    df          : DataFrame with at minimum a 'symptom_text' column.
                  Optional columns: disease_encoded (if has_disease=True),
                  risk_vector (if has_risk=True), severity_score,
                  urgency_encoded, specialist_encoded.
    tokenizer   : HuggingFace tokenizer
    max_len     : token length limit (default 256, same as original MedDataset)
    has_disease : set True when df contains disease_encoded — enables disease label output
    has_risk    : set True when df contains risk_vector — enables risk label output
    """
    def __init__(self, df, tokenizer, max_len=256, has_disease=False, has_risk=False):
        self.df          = df.reset_index(drop=True)
        self.tokenizer   = tokenizer
        self.max_len     = max_len
        self.has_disease = has_disease
        self.has_risk    = has_risk

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # ── Tokenise symptom text ──────────────────────────────────────────────
        enc = self.tokenizer(
            str(row['symptom_text']),
            max_length     = self.max_len,
            padding        = 'max_length',
            truncation     = True,
            return_tensors = 'pt'          # fixed: was misspelled in previous version
        )
        item = {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
        }

        # ── Original labels (only present in HCM-sourced rows) ────────────────
        if 'severity_score' in row.index:
            item['severity_cls'] = torch.tensor(int(row['severity_score']) - 1,         dtype=torch.long)
            item['severity_reg'] = torch.tensor((float(row['severity_score']) - 1) / 9, dtype=torch.float)
        if 'urgency_encoded' in row.index:
            item['urgency']      = torch.tensor(int(row['urgency_encoded']),             dtype=torch.long)
        if 'specialist_encoded' in row.index:
            item['specialist']   = torch.tensor(int(row['specialist_encoded']),          dtype=torch.long)

        # ── New: disease label (single class, 41-way) ─────────────────────────
        if self.has_disease and 'disease_encoded' in row.index:
            item['disease'] = torch.tensor(int(row['disease_encoded']), dtype=torch.long)

        # ── New: risk factor vector (10-dim multi-label binary) ───────────────
        if self.has_risk and 'risk_vector' in row.index:
            rv = row['risk_vector']
            if isinstance(rv, str):
                import ast
                rv = ast.literal_eval(rv)
            item['risk_vector'] = torch.tensor(rv, dtype=torch.float)

        return item


# ── Build disease training DataLoaders ───────────────────────────────────────
print('Building disease training split from HuggingFace data...')

# Stratified split — with fallback for classes that have too few samples
try:
    # Only stratify if every class has at least 2 samples
    min_count = df_disease['disease_encoded'].value_counts().min()
    if min_count >= 2:
        disease_train_df, disease_val_df = train_test_split(
            df_disease, test_size=0.15, random_state=42,
            stratify=df_disease['disease_encoded']
        )
        print(f'  Stratified split (min class size: {min_count})')
    else:
        raise ValueError(f'Class too small for stratify (min={min_count})')
except ValueError as e:
    print(f'  Falling back to random split: {e}')
    disease_train_df, disease_val_df = train_test_split(
        df_disease, test_size=0.15, random_state=42
    )

disease_train_ds     = MedDatasetExtended(disease_train_df, tokenizer,
                                           has_disease=True, has_risk=False)
disease_val_ds       = MedDatasetExtended(disease_val_df,   tokenizer,
                                           has_disease=True, has_risk=False)
disease_train_loader = DataLoader(disease_train_ds, batch_size=16, shuffle=True,
                                   num_workers=2, pin_memory=True)
disease_val_loader   = DataLoader(disease_val_ds,   batch_size=32, shuffle=False,
                                   num_workers=2, pin_memory=True)

# ── Build risk factor training DataLoaders ────────────────────────────────────
print('Building risk factor training split from HealthCareMagic data...')
df_risk_labeled = df_risk[df_risk['risk_vector'].apply(lambda v: sum(v) > 0)].copy()

if len(df_risk_labeled) < 10:
    print('WARNING: Very few risk-labeled rows. Risk head will train on all HCM data.')
    df_risk_labeled = df_risk.copy()

risk_train_df, risk_val_df = train_test_split(df_risk_labeled, test_size=0.15, random_state=42)
risk_train_ds     = MedDatasetExtended(risk_train_df, tokenizer,
                                        has_disease=False, has_risk=True)
risk_val_ds       = MedDatasetExtended(risk_val_df,   tokenizer,
                                        has_disease=False, has_risk=True)
risk_train_loader = DataLoader(risk_train_ds, batch_size=16, shuffle=True,
                                num_workers=2, pin_memory=True)
risk_val_loader   = DataLoader(risk_val_ds,   batch_size=32, shuffle=False,
                                num_workers=2, pin_memory=True)

print(f'\nDisease — Train: {len(disease_train_df):,}  Val: {len(disease_val_df):,}')
print(f'Risk    — Train: {len(risk_train_df):,}  Val: {len(risk_val_df):,}')
print(f'NUM_DISEASE = {NUM_DISEASE}')
print('Extended datasets ready.')


## Cell 7 — Model Architecture
Bio_ClinicalBERT backbone (110M params, 12 transformer layers, 768 hidden dims)
with four task-specific output heads attached to the pooler output.
The specialist_head has 3 layers because it has the hardest task (12 classes).

In [ ]:
class ClinicalBERTModel(nn.Module):
    def __init__(self, num_urgency, num_specialist):
        super().__init__()
        self.bert    = AutoModel.from_pretrained(BASE_MODEL)
        self.dropout = nn.Dropout(0.3)
        self.specialist_head = nn.Sequential(
            nn.Linear(768, 512), nn.ReLU(), nn.Dropout(0.25),
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, num_specialist)
        )
        self.urgency_head = nn.Sequential(
            nn.Linear(768, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, num_urgency)
        )
        self.severity_cls = nn.Sequential(
            nn.Linear(768, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 10)
        )
        self.severity_reg = nn.Sequential(
            nn.Linear(768, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 1), nn.Sigmoid()
        )
    def forward(self, input_ids, attention_mask):
        pooled = self.dropout(
            self.bert(input_ids=input_ids, attention_mask=attention_mask).pooler_output)
        return {
            'severity_cls': self.severity_cls(pooled),
            'severity_reg': self.severity_reg(pooled).squeeze(-1),
            'urgency':      self.urgency_head(pooled),
            'specialist':   self.specialist_head(pooled),
        }
print('ClinicalBERTModel defined.')

## Cell 7B — Extended Model with Disease + Risk Factor Heads

Adds two new output heads to the existing `ClinicalBERTModel` architecture:

### New heads
| Head | Type | Output size | Loss |
|------|------|-------------|------|
| `disease_head` | Multi-class | `NUM_DISEASE` classes (dynamic) | CrossEntropyLoss |
| `risk_head` | Multi-label | 10 risk factors | BCEWithLogitsLoss |

### Architecture notes
- `NUM_DISEASE` is set automatically from `disease_enc.classes_` in Cell 5C —
  so the model correctly handles 41, 392, or any other number of disease classes
  without manual changes
- `disease_head` uses 3 layers (768→512→256→N) — same depth as `specialist_head`
- `risk_head` uses 2 layers (768→256→10); outputs **raw logits** — sigmoid is
  applied at inference time only (BCEWithLogitsLoss handles it internally during training)
- Both heads share the same BERT backbone as all existing heads — zero extra BERT params

### Weight loading strategy
We load the Stage 3 checkpoint with `strict=False`:
- BERT + existing head weights → loaded from Stage 3 (preserves all learned representations)
- `disease_head` + `risk_head` weights → random init (trained fresh in Stage 4)


In [ ]:
class ClinicalBERTModelExtended(nn.Module):
    """
    Bio_ClinicalBERT with six task-specific output heads:

    Existing (trained in Stages 1-3):
      - specialist_head  : 12-class specialist classification
      - urgency_head     : 4-class urgency classification
      - severity_cls     : 10-class ordinal severity
      - severity_reg     : regression severity (0-1 normalised)

    New (trained in Stage 4):
      - disease_head     : 41-class disease classification
      - risk_head        : 10-label binary risk factor classification

    The BERT backbone is shared across all six heads — all tasks benefit
    from the same rich clinical language representations.
    """
    def __init__(self, num_urgency, num_specialist, num_disease, num_risk=10):
        super().__init__()

        # ── Shared BERT backbone ──────────────────────────────────────────────
        self.bert    = AutoModel.from_pretrained(BASE_MODEL)
        self.dropout = nn.Dropout(0.3)

        # ── Existing heads (unchanged from original model) ────────────────────
        self.specialist_head = nn.Sequential(
            nn.Linear(768, 512), nn.ReLU(), nn.Dropout(0.25),
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, num_specialist)
        )
        self.urgency_head = nn.Sequential(
            nn.Linear(768, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, num_urgency)
        )
        self.severity_cls = nn.Sequential(
            nn.Linear(768, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 10)
        )
        self.severity_reg = nn.Sequential(
            nn.Linear(768, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 1), nn.Sigmoid()
        )

        # ── New: Disease prediction head ──────────────────────────────────────
        # 3-layer MLP — same depth as specialist_head (41 classes is hard)
        self.disease_head = nn.Sequential(
            nn.Linear(768, 512), nn.ReLU(), nn.Dropout(0.25),
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, num_disease)
        )

        # ── New: Risk factor head (multi-label) ───────────────────────────────
        # Outputs raw logits — BCEWithLogitsLoss handles sigmoid internally.
        # At inference, apply sigmoid manually and threshold at 0.5.
        self.risk_head = nn.Sequential(
            nn.Linear(768, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, num_risk)
        )

    def forward(self, input_ids, attention_mask):
        pooled = self.dropout(
            self.bert(input_ids=input_ids, attention_mask=attention_mask).pooler_output
        )
        return {
            # Existing outputs
            'severity_cls': self.severity_cls(pooled),
            'severity_reg': self.severity_reg(pooled).squeeze(-1),
            'urgency':      self.urgency_head(pooled),
            'specialist':   self.specialist_head(pooled),
            # New outputs
            'disease':      self.disease_head(pooled),
            'risk':         self.risk_head(pooled),
        }


# ── Build extended model and load Stage 3 weights ────────────────────────────
print('Building ClinicalBERTModelExtended...')
model_ext = ClinicalBERTModelExtended(
    num_urgency    = NUM_URGENCY,
    num_specialist = NUM_SPECIALIST,
    num_disease    = NUM_DISEASE,
    num_risk       = len(RISK_FACTOR_LABELS)
).to(device)

# Load Stage 3 best weights into matching layers
best_s3 = os.path.join(DRIVE_DIR, 'best_stage3.pth')
if os.path.exists(best_s3):
    s3_weights = torch.load(best_s3, map_location=device)
    # strict=False: only loads keys that exist in both dicts
    # New heads (disease_head, risk_head) start from random init
    missing, unexpected = model_ext.load_state_dict(s3_weights, strict=False)
    print(f'Loaded Stage 3 weights. Missing keys (new heads, expected): {len(missing)}')
    print(f'Unexpected keys: {len(unexpected)}')
else:
    print('WARNING: best_stage3.pth not found. Starting from scratch.')
    print('         Run Stages 1-3 first for best results.')

total  = sum(p.numel() for p in model_ext.parameters())
bert_p = sum(p.numel() for p in model_ext.bert.parameters())
new_p  = (sum(p.numel() for p in model_ext.disease_head.parameters()) +
          sum(p.numel() for p in model_ext.risk_head.parameters()))
print(f'\nTotal parameters   : {total:,}')
print(f'BERT parameters    : {bert_p:,}')
print(f'New head parameters: {new_p:,}')
print('Model ready.')


## Cell 8 — Loss Functions & Training Helpers

### Loss function decisions
- **`spec_loss_fn`** — plain CrossEntropyLoss, no class weights.
  Dataset is perfectly balanced (3,000/class) so weighting would hurt, not help.
- **`urg_loss_fn`** — weighted CrossEntropyLoss.
  Urgency imbalance survives specialist rebalancing: routine ~47%, immediate ~9%.
  Weights ensure 'immediate' gets learned even though it is rarer.
- **`sev_cls_fn`** — CrossEntropyLoss with label_smoothing=0.1.
  Score-4 still dominates severity distribution — smoothing softens this.
- **`mse_loss`** — MSELoss on 0-1 normalised severity regression.

### Per-epoch monitoring
`evaluate()` reports per-class accuracy for both heads every epoch:
- Watch for specialist classes stuck at 0.00 (model not learning that class)
- Watch for `immediate=0.00` in urgency (clinically dangerous — must fix)
- After Stage 3, all specialist classes should be >0.60, immediate >0.65

In [ ]:
def compute_class_weights(df_split, label_col, encoder, device):
    """Standard sklearn formula: weight[i] = total / (n_classes * count[i])"""
    counts  = df_split[label_col].value_counts()
    n_total = len(df_split)
    n_class = len(encoder.classes_)
    weights = [n_total / (n_class * counts.get(cls, 1)) for cls in encoder.classes_]
    w = torch.tensor(weights, dtype=torch.float).to(device)
    for cls, wt in zip(encoder.classes_, weights):
        print(f'    {cls:<20} {wt:.3f}')
    return w

print('Urgency class weights (from balanced train split):')
urg_weights  = compute_class_weights(train_df, 'urgency', urgency_enc, device)

spec_loss_fn = nn.CrossEntropyLoss()
urg_loss_fn  = nn.CrossEntropyLoss(weight=urg_weights)
sev_cls_fn   = nn.CrossEntropyLoss(label_smoothing=0.1)
mse_loss     = nn.MSELoss()
print('\nLoss functions ready.')


def evaluate(model, loader):
    model.eval()
    sp, st, up, ut, srp, srt, reg_err = [], [], [], [], [], [], []
    with torch.no_grad():
        for b in tqdm(loader, desc='Eval', leave=False):
            ids, mask = b['input_ids'].to(device), b['attention_mask'].to(device)
            out = model(ids, mask)
            sp.extend(torch.argmax(out['specialist'],    1).cpu().tolist())
            st.extend(b['specialist'].tolist())
            up.extend(torch.argmax(out['urgency'],       1).cpu().tolist())
            ut.extend(b['urgency'].tolist())
            srp.extend(torch.argmax(out['severity_cls'], 1).cpu().tolist())
            srt.extend(b['severity_cls'].tolist())
            reg_err.extend((out['severity_reg'].cpu() - b['severity_reg']).abs().tolist())
    metrics = {
        'specialist_acc': accuracy_score(st, sp),
        'urgency_acc':    accuracy_score(ut, up),
        'severity_acc':   accuracy_score(srt, srp),
        'severity_mae':   float(np.mean(reg_err)) * 9,
    }
    spec_pc = {}
    for i, cls in enumerate(specialist_enc.classes_):
        idxs = [j for j, t in enumerate(st) if t == i]
        if idxs: spec_pc[cls] = sum(1 for j in idxs if sp[j] == i) / len(idxs)
    metrics['spec_per_class'] = spec_pc
    urg_pc = {}
    for i, cls in enumerate(urgency_enc.classes_):
        idxs = [j for j, t in enumerate(ut) if t == i]
        if idxs: urg_pc[cls] = sum(1 for j in idxs if up[j] == i) / len(idxs)
    metrics['urg_per_class'] = urg_pc
    return metrics


def print_metrics(metrics, stage, epoch):
    print(f'[{stage}] E{epoch} | '
          f'spec={metrics["specialist_acc"]:.4f} '
          f'urg={metrics["urgency_acc"]:.4f} '
          f'sev_acc={metrics["severity_acc"]:.4f} '
          f'sev_mae={metrics["severity_mae"]:.2f}')
    spec_pc  = metrics['spec_per_class']
    low_spec = {k: v for k, v in spec_pc.items() if v < 0.50}
    if low_spec:
        print('  WARNING spec <50%: ' + ', '.join(
            f'{k}={v:.2f}' for k, v in sorted(low_spec.items(), key=lambda x: x[1])))
    else:
        worst = min(spec_pc.items(), key=lambda x: x[1])
        print(f'  Spec worst: {worst[0]}={worst[1]:.2f}')
    urg_pc  = metrics['urg_per_class']
    imm_acc = urg_pc.get('immediate', 0)
    flag    = '  WARNING CRITICAL — missing emergencies' if imm_acc < 0.60 else ''
    print('  Urgency: ' + ' | '.join(f'{k}={v:.2f}' for k, v in urg_pc.items()) + flag)


def save_checkpoint(stage, epoch, model, optimizer, best_acc, history):
    """Save full training state — survives session restarts."""
    torch.save({
        'stage': stage, 'epoch': epoch,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'best_acc': best_acc, 'history': history,
    }, os.path.join(DRIVE_DIR, f'ckpt_{stage}.pth'))


def load_checkpoint(stage, model, optimizer):
    """Load checkpoint if exists. Returns (start_epoch, best_acc, history)."""
    path = os.path.join(DRIVE_DIR, f'ckpt_{stage}.pth')
    if not os.path.exists(path): return 0, 0.0, []
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    print(f'  Resumed {stage} from epoch {ckpt["epoch"]+1}, best_acc={ckpt["best_acc"]:.4f}')
    return ckpt['epoch'] + 1, ckpt['best_acc'], ckpt['history']


def run_stage(stage, epochs, optimizer, scheduler, loss_fn, patience=3):
    """
    Train one stage with per-epoch checkpointing and early stopping.
    If session drops mid-epoch, re-running this cell resumes from the last
    completed epoch automatically.
    """
    start_epoch, best_acc, history = load_checkpoint(stage, model, optimizer)
    no_improve = 0
    for epoch in range(start_epoch, epochs):
        model.train(); total_loss = 0.0
        for b in tqdm(train_loader, desc=f'{stage} E{epoch+1}/{epochs}'):
            ids, mask = b['input_ids'].to(device), b['attention_mask'].to(device)
            loss = loss_fn(model(ids, mask), b)
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            if scheduler: scheduler.step()
            total_loss += loss.item()
        metrics  = evaluate(model, val_loader)
        avg_loss = total_loss / len(train_loader)
        print_metrics(metrics, stage, f'{epoch+1}/{epochs} loss={avg_loss:.4f}')
        history.append({
            'stage': stage, 'epoch': epoch+1, 'loss': avg_loss,
            'specialist_acc': metrics['specialist_acc'],
            'urgency_acc':    metrics['urgency_acc'],
            'severity_acc':   metrics['severity_acc'],
            'severity_mae':   metrics['severity_mae'],
        })
        if metrics['specialist_acc'] > best_acc:
            best_acc = metrics['specialist_acc']; no_improve = 0
            torch.save(model.state_dict(), os.path.join(DRIVE_DIR, f'best_{stage}.pth'))
            print(f'  Saved best (spec_acc={best_acc:.4f})')
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  Early stopping at epoch {epoch+1}'); break
        save_checkpoint(stage, epoch, model, optimizer, best_acc, history)
    best_path = os.path.join(DRIVE_DIR, f'best_{stage}.pth')
    if os.path.exists(best_path):
        model.load_state_dict(torch.load(best_path, map_location=device))
    print(f'  [{stage}] Complete. Best spec_acc: {best_acc:.4f}')
    return history

print('\nAll helpers loaded.')

## Cell 9 — Tokenizer, DataLoaders, Model
Loads Bio_ClinicalBERT tokenizer and builds the model.
If a download error occurs here, re-run Cell 0C to clear the cache first.

batch_size=16 fits T4 (15GB VRAM) with max_len=256. Reduce to 8 for OOM errors.

In [ ]:
tokenizer    = AutoTokenizer.from_pretrained(BASE_MODEL)
train_ds     = MedDataset(train_df, tokenizer)
val_ds       = MedDataset(val_df,   tokenizer)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train batches : {len(train_loader):,}  Val batches: {len(val_loader):,}')

model = ClinicalBERTModel(NUM_URGENCY, NUM_SPECIALIST).to(device)
print(f'Total params  : {sum(p.numel() for p in model.parameters()):,}')
print(f'BERT params   : {sum(p.numel() for p in model.bert.parameters()):,}')
all_history = []

## Cell 10 — Stage 1: Specialist Head Only (BERT frozen)

BERT weights are frozen — only the specialist_head (randomly initialised) learns.
Freezing BERT prevents random head gradients from corrupting its pre-trained representations.

**3 epochs only** — previous run with this dataset plateaued at epoch 5, so 3 epochs
captures the same result in ~24 min instead of ~1.75 hrs.

**Expected result:** spec_acc ~0.35-0.42 (plateau before BERT fine-tunes in Stage 3)

⏱️ ~24 min

In [ ]:
print('='*60)
print('STAGE 1: specialist_head only — BERT frozen')
print('Expected: spec_acc ~0.35-0.42 (Stage 3 will jump to 0.80+)')
print('='*60)
for p in model.bert.parameters(): p.requires_grad = False

opt1 = AdamW(model.specialist_head.parameters(), lr=1e-3, weight_decay=0.01)
sch1 = get_linear_schedule_with_warmup(
    opt1,
    num_warmup_steps=50,
    num_training_steps=len(train_loader) * 3
)
h = run_stage('stage1', epochs=3, optimizer=opt1, scheduler=sch1,
              loss_fn=lambda out, b: spec_loss_fn(out['specialist'], b['specialist'].to(device)))
all_history.extend(h)

## Cell 11 — Stage 2: Urgency Head Only (BERT frozen)

BERT still frozen. Urgency head learns independently with weighted loss.
The weighted loss forces the model to learn 'immediate' despite it being only ~9% of data.

**3 epochs only** — previous run early-stopped at epoch 4, so 3 epochs is the natural end.

**Expected result:** urg_acc ~0.50-0.56, immediate recall ~0.30-0.45

⏱️ ~24 min

In [ ]:
print('='*60)
print('STAGE 2: urgency_head only — BERT frozen')
print('Expected: immediate recall ~0.30-0.45 (Stage 3 will improve further)')
print('='*60)
opt2 = AdamW(model.urgency_head.parameters(), lr=1e-3, weight_decay=0.01)
sch2 = get_linear_schedule_with_warmup(
    opt2,
    num_warmup_steps=50,
    num_training_steps=len(train_loader) * 3
)
h = run_stage('stage2', epochs=3, optimizer=opt2, scheduler=sch2,
              loss_fn=lambda out, b: urg_loss_fn(out['urgency'], b['urgency'].to(device)))
all_history.extend(h)

## Cell 11B — Verify Checkpoints Before Stage 3
Run this before starting Stage 3 to confirm Stage 1 and 2 weights are saved.
If any file shows MISSING, do not proceed — re-run the corresponding stage.

In [ ]:
print('Checkpoint verification:')
needed = ['best_stage1.pth', 'ckpt_stage1.pth',
          'best_stage2.pth', 'ckpt_stage2.pth',
          'combined_dataset_v2.csv', 'urgency_encoder.pkl', 'specialist_encoder.pkl']
all_ok = True
for f in needed:
    path   = os.path.join(DRIVE_DIR, f)
    exists = os.path.exists(path)
    size   = f'{os.path.getsize(path)/1e6:.1f}MB' if exists else 'MISSING'
    status = 'OK' if exists else 'MISSING'
    if not exists: all_ok = False
    print(f'  {status:<7}  {size:<10}  {f}')
print()
if all_ok:
    print('All files present. Safe to run Stage 3.')
else:
    print('WARNING: Missing files detected. Re-run the relevant stage before Stage 3.')

## Cell 12 — Stage 3: Joint Fine-Tuning (All Layers Unfrozen)

This is where the real learning happens. BERT unfreezes and reshapes its internal
representations to serve your specific symptom classification task.

**Differential learning rates:**
- BERT encoder: 2e-5 (very low — adapts gently without forgetting medical language)
- All heads: 5e-5 (moderate — continues improving from Stage 1/2 starting point)
- BERT embeddings: frozen (rarely benefit from fine-tuning)

**Joint loss weights:** specialist (1.5) > urgency (1.2) > severity_cls (1.0) > severity_reg (0.5)

**Expected results after Stage 3:**
- spec_acc: ~0.87-0.93
- urg_acc:  ~0.88-0.93
- immediate recall: ~0.70-0.82
- All specialist classes: >0.65

**If session drops mid-Stage 3:** re-run Cells 0C, 1-9, then run THIS cell only.
It will auto-resume from the last completed epoch.

⏱️ ~3.5 hrs (10 epochs × ~22 min)

In [ ]:
print('='*60)
print('STAGE 3: joint fine-tune — all layers unfrozen')
print('Expected epoch 1: spec ~0.78-0.82 (large jump from Stage 1 plateau)')
print('='*60)
for p in model.bert.parameters():            p.requires_grad = True
for p in model.bert.embeddings.parameters(): p.requires_grad = False

opt3 = AdamW([
    {'params': model.bert.parameters(),            'lr': 2e-5},
    {'params': model.specialist_head.parameters(), 'lr': 5e-5},
    {'params': model.urgency_head.parameters(),    'lr': 5e-5},
    {'params': model.severity_cls.parameters(),    'lr': 5e-5},
    {'params': model.severity_reg.parameters(),    'lr': 5e-5},
], weight_decay=0.01)
total_steps = len(train_loader) * 10
sch3 = get_linear_schedule_with_warmup(
    opt3,
    num_warmup_steps=int(total_steps * 0.1),
    num_training_steps=total_steps
)

def joint_loss(out, b):
    return (
        1.5 * spec_loss_fn(out['specialist'],   b['specialist'].to(device))   +
        1.2 * urg_loss_fn(out['urgency'],       b['urgency'].to(device))      +
        1.0 * sev_cls_fn(out['severity_cls'],   b['severity_cls'].to(device)) +
        0.5 * mse_loss(out['severity_reg'],     b['severity_reg'].to(device))
    )

h = run_stage('stage3', epochs=5, optimizer=opt3, scheduler=sch3,
              loss_fn=joint_loss, patience=5)
all_history.extend(h)

## Cell 12B — Stage 4: Disease + Risk Factor Head Training

Trains the two new heads on top of the already fine-tuned BERT backbone.

### Strategy: BERT frozen
We keep BERT and all original heads frozen for Stage 4 because:
- BERT already has strong clinical representations from Stage 3
- Freezing prevents catastrophic forgetting of specialist/urgency accuracy
- Training only ~450K new parameters is ~240× faster than the full model

### Loss functions
- **Disease loss** — `CrossEntropyLoss` over `NUM_DISEASE` classes
  (the actual number from your HuggingFace dataset — set automatically)
- **Risk loss** — `BCEWithLogitsLoss` with `pos_weight` to upweight rare risk factors
  Each of the 10 risk factors is an independent binary classification

### Expected metrics
| Metric | Expected range | Meaning |
|--------|---------------|---------|
| `disease_acc` | 0.55 – 0.75 | Acceptable for top-3 display |
| `risk_f1` | 0.50 – 0.70 | Good enough for clinical hints |

Lower disease accuracy is expected because the HuggingFace dataset is small (~853 rows).
The top-3 output with confidence scores compensates — even a 60% accurate model gives
meaningful probabilistic guidance.

### Training time
~10–15 minutes for 5 epochs (BERT frozen, small dataset)


In [ ]:
print('='*60)
print('STAGE 4: disease_head + risk_head — BERT frozen')
print('Expected: disease_acc ~0.65-0.80  risk_f1 ~0.55-0.70')
print('='*60)

# ── Freeze BERT and existing heads — only train new heads ─────────────────────
for p in model_ext.bert.parameters():            p.requires_grad = False
for p in model_ext.specialist_head.parameters(): p.requires_grad = False
for p in model_ext.urgency_head.parameters():    p.requires_grad = False
for p in model_ext.severity_cls.parameters():    p.requires_grad = False
for p in model_ext.severity_reg.parameters():    p.requires_grad = False

# Only new heads are trainable
for p in model_ext.disease_head.parameters(): p.requires_grad = True
for p in model_ext.risk_head.parameters():    p.requires_grad = True

trainable = sum(p.numel() for p in model_ext.parameters() if p.requires_grad)
print(f'Trainable parameters: {trainable:,}  (disease + risk heads only)')

# ── Loss functions for new heads ──────────────────────────────────────────────
disease_loss_fn = nn.CrossEntropyLoss()

# Compute pos_weight for BCEWithLogitsLoss — upweights rare risk factors
risk_array  = np.array([row for row in df_risk_labeled['risk_vector'].tolist()])
pos_counts  = risk_array.sum(axis=0) + 1  # +1 smoothing
neg_counts  = len(risk_array) - pos_counts + 1
pos_weights = torch.tensor(neg_counts / pos_counts, dtype=torch.float).to(device)
risk_loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

print('\nRisk factor pos_weights (higher = rarer = upweighted):')
for label, pw in zip(RISK_FACTOR_LABELS, pos_weights.cpu().tolist()):
    print(f'  {label:<22} {pw:.2f}')

# ── Optimiser — only new head parameters ─────────────────────────────────────
opt4 = AdamW([
    {'params': model_ext.disease_head.parameters(), 'lr': 1e-3},
    {'params': model_ext.risk_head.parameters(),    'lr': 1e-3},
], weight_decay=0.01)

NUM_EPOCHS_S4 = 5
sch4 = get_linear_schedule_with_warmup(
    opt4,
    num_warmup_steps    = 50,
    num_training_steps  = (len(disease_train_loader) + len(risk_train_loader)) * NUM_EPOCHS_S4
)

# ── Training loop ─────────────────────────────────────────────────────────────
best_disease_acc = 0.0
stage4_history   = []

for epoch in range(NUM_EPOCHS_S4):
    model_ext.train()
    d_loss_total = r_loss_total = 0.0

    # Disease head training pass
    for b in tqdm(disease_train_loader, desc=f'S4 E{epoch+1} Disease'):
        ids, mask = b['input_ids'].to(device), b['attention_mask'].to(device)
        out   = model_ext(ids, mask)
        loss  = disease_loss_fn(out['disease'], b['disease'].to(device))
        opt4.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model_ext.parameters(), 1.0)
        opt4.step(); sch4.step()
        d_loss_total += loss.item()

    # Risk head training pass
    for b in tqdm(risk_train_loader, desc=f'S4 E{epoch+1} Risk'):
        ids, mask = b['input_ids'].to(device), b['attention_mask'].to(device)
        out   = model_ext(ids, mask)
        loss  = risk_loss_fn(out['risk'], b['risk_vector'].to(device))
        opt4.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model_ext.parameters(), 1.0)
        opt4.step(); sch4.step()
        r_loss_total += loss.item()

    # ── Evaluation ────────────────────────────────────────────────────────────
    model_ext.eval()
    d_preds, d_true = [], []
    r_preds, r_true = [], []

    with torch.no_grad():
        for b in disease_val_loader:
            ids, mask = b['input_ids'].to(device), b['attention_mask'].to(device)
            out = model_ext(ids, mask)
            d_preds.extend(torch.argmax(out['disease'], 1).cpu().tolist())
            d_true.extend(b['disease'].tolist())

        for b in risk_val_loader:
            ids, mask = b['input_ids'].to(device), b['attention_mask'].to(device)
            out = model_ext(ids, mask)
            probs = torch.sigmoid(out['risk']).cpu().numpy()
            r_preds.extend((probs > 0.5).astype(int).tolist())
            r_true.extend(b['risk_vector'].numpy().astype(int).tolist())

    disease_acc = accuracy_score(d_true, d_preds)

    # Micro F1 for multi-label
    from sklearn.metrics import f1_score
    risk_f1 = f1_score(np.array(r_true), np.array(r_preds), average='micro', zero_division=0)

    avg_d = d_loss_total / len(disease_train_loader)
    avg_r = r_loss_total / len(risk_train_loader)
    print(f'[Stage4] E{epoch+1}/{NUM_EPOCHS_S4} | '
          f'd_loss={avg_d:.4f} r_loss={avg_r:.4f} | '
          f'disease_acc={disease_acc:.4f} risk_f1={risk_f1:.4f}')

    stage4_history.append({
        'epoch': epoch+1, 'disease_acc': disease_acc, 'risk_f1': float(risk_f1),
        'd_loss': avg_d, 'r_loss': avg_r
    })

    if disease_acc > best_disease_acc:
        best_disease_acc = disease_acc
        torch.save(model_ext.state_dict(), os.path.join(DRIVE_DIR, 'best_stage4.pth'))
        print(f'  Saved best (disease_acc={best_disease_acc:.4f})')

print(f'\nStage 4 complete. Best disease_acc: {best_disease_acc:.4f}')

# Load best weights
model_ext.load_state_dict(torch.load(os.path.join(DRIVE_DIR, 'best_stage4.pth'), map_location=device))
print('Loaded best Stage 4 weights.')


In [ ]:
print('='*60)
print('STAGE 4: disease_head + risk_head — BERT frozen')
print('Continued training: 10 epochs, lower LR (3e-4)')
print('='*60)

# ── Freeze BERT and existing heads — only train new heads ─────────────────────
for p in model_ext.bert.parameters():            p.requires_grad = False
for p in model_ext.specialist_head.parameters(): p.requires_grad = False
for p in model_ext.urgency_head.parameters():    p.requires_grad = False
for p in model_ext.severity_cls.parameters():    p.requires_grad = False
for p in model_ext.severity_reg.parameters():    p.requires_grad = False

for p in model_ext.disease_head.parameters(): p.requires_grad = True
for p in model_ext.risk_head.parameters():    p.requires_grad = True

trainable = sum(p.numel() for p in model_ext.parameters() if p.requires_grad)
print(f'Trainable parameters: {trainable:,}  (disease + risk heads only)')

# ── Loss functions ─────────────────────────────────────────────────────────────
disease_loss_fn = nn.CrossEntropyLoss()

risk_array   = np.array([row for row in df_risk_labeled['risk_vector'].tolist()])
pos_counts   = risk_array.sum(axis=0) + 1
neg_counts   = len(risk_array) - pos_counts + 1
pos_weights  = torch.tensor(neg_counts / pos_counts, dtype=torch.float).to(device)
# Cap pos_weights at 20 — sedentary(106) and poor_diet(182) were too dominant
pos_weights  = torch.clamp(pos_weights, max=20.0)
risk_loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

print('\nRisk factor pos_weights (capped at 20):')
for label, pw in zip(RISK_FACTOR_LABELS, pos_weights.cpu().tolist()):
    print(f'  {label:<22} {pw:.2f}')

# ── Load best weights from previous run to continue from epoch 5 ───────────────
prev_best = os.path.join(DRIVE_DIR, 'best_stage4.pth')
if os.path.exists(prev_best):
    model_ext.load_state_dict(torch.load(prev_best, map_location=device))
    print('\nResuming from previous best Stage 4 weights (epoch 5, acc=0.5625)')
else:
    print('\nNo previous Stage 4 checkpoint — starting fresh')
    # ── Optimiser — lower LR since we're continuing ───────────────────────────────
NUM_EPOCHS_S4 = 10
opt4 = AdamW([
    {'params': model_ext.disease_head.parameters(), 'lr': 3e-4},
    {'params': model_ext.risk_head.parameters(),    'lr': 3e-4},
], weight_decay=0.01)

sch4 = get_linear_schedule_with_warmup(
    opt4,
    num_warmup_steps   = 20,
    num_training_steps = (len(disease_train_loader) + len(risk_train_loader)) * NUM_EPOCHS_S4
)

# ── Training loop ──────────────────────────────────────────────────────────────
best_disease_acc = 0.5625   # carry over from previous run
no_improve       = 0
PATIENCE         = 4
stage4_history   = []

for epoch in range(NUM_EPOCHS_S4):
    model_ext.train()
    d_loss_total = r_loss_total = 0.0

    # Disease head pass
    for b in tqdm(disease_train_loader, desc=f'S4 E{epoch+1} Disease'):
        ids, mask = b['input_ids'].to(device), b['attention_mask'].to(device)
        out  = model_ext(ids, mask)
        loss = disease_loss_fn(out['disease'], b['disease'].to(device))
        opt4.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model_ext.parameters(), 1.0)
        opt4.step(); sch4.step()
        d_loss_total += loss.item()
    
    # Risk head pass
    for b in tqdm(risk_train_loader, desc=f'S4 E{epoch+1} Risk  '):
        ids, mask = b['input_ids'].to(device), b['attention_mask'].to(device)
        out  = model_ext(ids, mask)
        loss = risk_loss_fn(out['risk'], b['risk_vector'].to(device))
        opt4.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model_ext.parameters(), 1.0)
        opt4.step(); sch4.step()
        r_loss_total += loss.item()

    # ── Evaluation ────────────────────────────────────────────────────────────
    model_ext.eval()
    d_preds, d_true = [], []
    r_preds, r_true = [], []

    with torch.no_grad():
        for b in disease_val_loader:
            ids, mask = b['input_ids'].to(device), b['attention_mask'].to(device)
            out = model_ext(ids, mask)
            d_preds.extend(torch.argmax(out['disease'], 1).cpu().tolist())
            d_true.extend(b['disease'].tolist())

        for b in risk_val_loader:
            ids, mask = b['input_ids'].to(device), b['attention_mask'].to(device)
            out   = model_ext(ids, mask)
            probs = torch.sigmoid(out['risk']).cpu().numpy()
            r_preds.extend((probs > 0.5).astype(int).tolist())
            r_true.extend(b['risk_vector'].numpy().astype(int).tolist())

    from sklearn.metrics import f1_score, classification_report
    disease_acc = accuracy_score(d_true, d_preds)
    risk_f1     = f1_score(np.array(r_true), np.array(r_preds),
                           average='micro', zero_division=0)

    # Top-3 accuracy — more meaningful metric for disease head
    def top3_acc(preds_logits_list, true_list):
        correct = 0
        model_ext.eval()
        with torch.no_grad():
            for b in disease_val_loader:
                ids, mask = b['input_ids'].to(device), b['attention_mask'].to(device)
                out    = model_ext(ids, mask)
                probs  = torch.softmax(out['disease'], dim=1)
                top3   = torch.topk(probs, 3, dim=1).indices.cpu().tolist()
                labels = b['disease'].tolist()
                for t3, lbl in zip(top3, labels):
                    if lbl in t3: correct += 1
        return correct / len(disease_val_loader.dataset)

    top3 = top3_acc(d_preds, d_true)

    avg_d = d_loss_total / len(disease_train_loader)
    avg_r = r_loss_total / len(risk_train_loader)

    print(f'[Stage4] E{epoch+1:02d}/{NUM_EPOCHS_S4} | '
          f'd_loss={avg_d:.4f} r_loss={avg_r:.4f} | '
          f'disease_acc={disease_acc:.4f} top3={top3:.4f} risk_f1={risk_f1:.4f}')

    stage4_history.append({
        'epoch': epoch + 1,
        'disease_acc': round(disease_acc, 4),
        'top3_acc':    round(top3, 4),
        'risk_f1':     round(float(risk_f1), 4),
        'd_loss':      round(avg_d, 4),
        'r_loss':      round(avg_r, 4),
    })
    if disease_acc > best_disease_acc:
        best_disease_acc = disease_acc
        no_improve = 0
        torch.save(model_ext.state_dict(), os.path.join(DRIVE_DIR, 'best_stage4.pth'))
        print(f'  ✓ Saved best  disease_acc={best_disease_acc:.4f}  top3={top3:.4f}')
    else:
        no_improve += 1
        print(f'  No improvement ({no_improve}/{PATIENCE})')
        if no_improve >= PATIENCE:
            print(f'  Early stopping at epoch {epoch+1}')
            break

# ── Final summary ──────────────────────────────────────────────────────────────
print(f'\nStage 4 complete.')
print(f'  Best top-1 disease_acc : {best_disease_acc:.4f}')
print(f'  Final top-3 accuracy   : {top3:.4f}')
print(f'  Final risk F1          : {risk_f1:.4f}')

# Load best weights
model_ext.load_state_dict(
    torch.load(os.path.join(DRIVE_DIR, 'best_stage4.pth'), map_location=device)
)
print('Loaded best Stage 4 weights.')
# Load best weights
model_ext.load_state_dict(
    torch.load(os.path.join(DRIVE_DIR, 'best_stage4.pth'), map_location=device)
)
print('Loaded best Stage 4 weights.')

# ── Per-risk-factor breakdown ──────────────────────────────────────────────────
print('\nPer-risk-factor F1 (to see which factors are weak):')
from sklearn.metrics import f1_score as f1_per
r_true_arr  = np.array(r_true)
r_preds_arr = np.array(r_preds)
for i, label in enumerate(RISK_FACTOR_LABELS):
    if r_true_arr[:, i].sum() > 0:
        f = f1_score(r_true_arr[:, i], r_preds_arr[:, i], zero_division=0)
        support = int(r_true_arr[:, i].sum())
        print(f'  {label:<22} F1={f:.3f}  support={support}')
    else:
        print(f'  {label:<22} F1=N/A   support=0  (never appears in val set)')

## Cell 13 — Save Final Model
Saves the best weights and prints the complete per-class accuracy report.
All specialist classes should be >0.65 and immediate urgency >0.65 before deploying.

In [ ]:
torch.save(model.state_dict(), os.path.join(DRIVE_DIR, 'biobert_clinical_best.pth'))
with open(os.path.join(DRIVE_DIR, 'training_history.json'), 'w') as f:
    json.dump(all_history, f, indent=2)

print('Final validation metrics:')
final = evaluate(model, val_loader)
for k, v in final.items():
    if k not in ('spec_per_class', 'urg_per_class'): print(f'  {k}: {v:.4f}')

print('\nPer-class specialist accuracy (sorted worst to best):')
for cls, acc in sorted(final['spec_per_class'].items(), key=lambda x: x[1]):
    flag = '  <- needs improvement' if acc < 0.65 else ''
    print(f'  {cls:<25} {acc:.4f}{flag}')

print('\nPer-class urgency accuracy:')
for cls, acc in final['urg_per_class'].items():
    flag = '  <- WARNING: missing emergencies' if cls == 'immediate' and acc < 0.65 else ''
    print(f'  {cls:<20} {acc:.4f}{flag}')

print(f'\nSaved to {DRIVE_DIR}')
print('Files to download from Kaggle Output tab:')
for f in ['biobert_clinical_best.pth', 'urgency_encoder.pkl', 'specialist_encoder.pkl']:
    size = os.path.getsize(os.path.join(DRIVE_DIR, f)) / 1e6
    print(f'  {size:.1f}MB  {f}')

## Cell 14 — Inference Test
7 known test cases. All should pass after Stage 3.
If any fail, check the per-class report in Cell 13 to identify the weak class.

In [ ]:
model.eval()
test_cases = [
    ('45M | chest pain radiating to left arm, sweating, short of breath', 'Cardiologist',       'immediate'),
    ('28F | fever 3 days, stomach pain, nausea, vomiting',                'Gastroenterologist', 'within_24_hours'),
    ('35M | severe headache, neck stiffness, sensitivity to light',       'Neurologist',        'within_24_hours'),
    ('22F | mild skin rash on arms, itching for 2 days',                  'Dermatologist',      'within_a_week'),
    ('60M | shortness of breath, wheezing, cough with mucus',             'Pulmonologist',      'within_a_week'),
    ('30F | anxiety, panic attacks, difficulty sleeping',                 'Psychiatrist',       'within_a_week'),
    ('50M | frequent urination, excessive thirst, weight loss',           'Urologist',          'within_a_week'),
]
correct_spec = correct_urg = 0
print('Inference test (7 cases):')
print('-'*70)
for text, exp_spec, exp_urg in test_cases:
    enc = tokenizer(text, return_tensors='pt', max_length=256,
                    truncation=True, padding='max_length')
    with torch.no_grad():
        out = model(enc['input_ids'].to(device), enc['attention_mask'].to(device))
    pred_spec = specialist_enc.classes_[torch.argmax(out['specialist']).item()]
    pred_urg  = urgency_enc.classes_[torch.argmax(out['urgency']).item()]
    pred_sev  = out['severity_reg'].item() * 9 + 1
    conf      = torch.softmax(out['specialist'], 1).max().item()
    s = '✓' if pred_spec == exp_spec else '✗'
    u = '✓' if pred_urg  == exp_urg  else '✗'
    if pred_spec == exp_spec: correct_spec += 1
    if pred_urg  == exp_urg:  correct_urg  += 1
    print(f'Input     : {text[:55]}')
    print(f'Specialist: {pred_spec} {s}  (expected {exp_spec}, conf={conf:.2f})')
    print(f'Urgency   : {pred_urg} {u}  (expected {exp_urg})')
    print(f'Severity  : {pred_sev:.1f}/10')
    print()
print(f'Result: Spec {correct_spec}/7  Urg {correct_urg}/7')
if correct_spec >= 6 and correct_urg >= 6:
    print('Model is ready for deployment.')
else:
    print('Some cases failed — check per-class report in Cell 13.')

## Cell 13B — Save Extended Model + Per-Disease Accuracy Report

Saves the final extended model weights and all encoders needed for inference.
Prints a full per-disease accuracy breakdown so you can see which diseases
the model predicts well and which need more data.

### Files produced
| File | Size (approx) | Purpose |
|------|--------------|---------|
| `biobert_clinical_extended.pth` | ~440MB | Full 6-head model weights |
| `disease_encoder.pkl` | <1KB | 41-class disease label encoder |
| `stage4_history.json` | <1KB | Stage 4 training metrics |

### Interpreting per-disease accuracy
- **>0.80** — strong; safe to show in top-1 prediction
- **0.60–0.80** — acceptable; show in top-3 with confidence score
- **<0.60** — weak; disease appears in top-3 but with low confidence visible to user
  (consider adding more training data for these classes)


In [ ]:
# ── Save extended model ──────────────────────────────────────────────────────
ext_path = os.path.join(DRIVE_DIR, 'biobert_clinical_extended.pth')
torch.save(model_ext.state_dict(), ext_path)
print(f'Saved extended model: {os.path.getsize(ext_path)/1e6:.1f}MB')

# Disease encoder already saved in Cell 5C — confirm it exists
assert os.path.exists(DISEASE_PKL), 'disease_encoder.pkl missing — re-run Cell 5C'
print(f'Disease encoder: {DISEASE_PKL}  OK')

# Save Stage 4 training history
with open(os.path.join(DRIVE_DIR, 'stage4_history.json'), 'w') as f:
    json.dump(stage4_history, f, indent=2)

# ── Per-disease accuracy report ───────────────────────────────────────────────
print('\n' + '='*60)
print('PER-DISEASE ACCURACY REPORT')
print('='*60)

model_ext.eval()
d_preds_all, d_true_all = [], []

with torch.no_grad():
    for b in disease_val_loader:
        ids, mask = b['input_ids'].to(device), b['attention_mask'].to(device)
        out = model_ext(ids, mask)
        d_preds_all.extend(torch.argmax(out['disease'], 1).cpu().tolist())
        d_true_all.extend(b['disease'].tolist())

per_disease = {}
for i, cls in enumerate(disease_enc.classes_):
    idxs = [j for j, t in enumerate(d_true_all) if t == i]
    if idxs:
        per_disease[cls] = sum(1 for j in idxs if d_preds_all[j] == i) / len(idxs)

print(f'  Overall disease accuracy: {accuracy_score(d_true_all, d_preds_all):.4f}\n')
print(f'  {"Disease":<40} {"Accuracy":>8}  Grade')
print(f'  {"-"*55}')
for cls, acc in sorted(per_disease.items(), key=lambda x: x[1]):
    grade = 'STRONG' if acc >= 0.80 else ('OK' if acc >= 0.60 else 'WEAK')
    flag  = '  <- needs more data' if acc < 0.60 else ''
    print(f'  {cls:<40} {acc:>7.4f}  {grade}{flag}')
print('='*60)

# ── Files summary ─────────────────────────────────────────────────────────────
print('\nFiles ready to download from Kaggle Output tab:')
for fname in ['biobert_clinical_extended.pth', 'disease_encoder.pkl',
              'urgency_encoder.pkl', 'specialist_encoder.pkl',
              'stage4_history.json']:
    full   = os.path.join(DRIVE_DIR, fname)
    status = f'{os.path.getsize(full)/1e6:.1f}MB' if os.path.exists(full) else 'MISSING'
    print(f'  {status:<10}  {fname}')


## Cell 14B — Full Swasthya Inference: All 6 Outputs

Demonstrates the complete Swasthya pipeline with all six outputs from a
single model call, plus the rule-based risk engine and knowledge map fusion.

### Output structure
```
Input: free-text symptom description

Output:
  Specialist       — from specialist_head  (existing)
  Urgency          — from urgency_head     (existing)
  Severity         — from severity_reg     (existing)
  Top-3 Diseases   — from disease_head     (NEW)
  Risk Factors     — BERT risk_head        (NEW)
                   + rule engine           (NEW)
                   + linked from map       (NEW)
  Questions        — from T5 model         (existing, run separately)
```

### How risk factors are merged
Three sources are combined and deduplicated:
1. **BERT risk_head** — soft learned signal from training data
2. **Rule engine** — regex patterns from Cell 5D (high precision)
3. **Knowledge map** — clinical risks linked to predicted diseases

This 3-layer fusion ensures comprehensive coverage:
- BERT catches implicit signals ("stressed at work" → stress risk)
- Rules catch explicit mentions ("smoker" → smoking_history)
- Knowledge map adds clinical context (Heart Attack → Hypertension risk)


In [ ]:
# ── Load extended model for inference ────────────────────────────────────────
model_ext.eval()

def predict_full(symptom_text, top_k=3, risk_threshold=0.45, verbose=True):
    """
    Full Swasthya inference pipeline.

    Parameters
    ----------
    symptom_text   : str  — free-text patient symptom description
    top_k          : int  — number of top disease predictions to return (default 3)
    risk_threshold : float — sigmoid threshold for BERT risk factor detection (default 0.45)
    verbose        : bool — whether to print formatted output

    Returns
    -------
    dict with keys: specialist, urgency, severity, diseases, risk_factors
    """
    # ── Tokenise ──────────────────────────────────────────────────────────────
    enc = tokenizer(symptom_text, return_tensors='pt', max_length=256,
                    truncation=True, padding='max_length')
    ids, mask = enc['input_ids'].to(device), enc['attention_mask'].to(device)

    with torch.no_grad():
        out = model_ext(ids, mask)

    # ── Existing predictions ──────────────────────────────────────────────────
    pred_specialist = specialist_enc.classes_[torch.argmax(out['specialist']).item()]
    pred_urgency    = urgency_enc.classes_[torch.argmax(out['urgency']).item()]
    pred_severity   = round(out['severity_reg'].item() * 9 + 1, 1)

    # ── Disease: top-K with confidence ───────────────────────────────────────
    disease_probs  = torch.softmax(out['disease'], dim=1).squeeze(0).cpu().numpy()
    top_k_idx      = disease_probs.argsort()[::-1][:top_k]
    top_diseases   = [
        {'disease': disease_enc.classes_[i], 'confidence': round(float(disease_probs[i]), 3)}
        for i in top_k_idx
    ]

    # ── Risk factors: 3-layer fusion ─────────────────────────────────────────
    # Layer 1: BERT risk_head
    risk_probs      = torch.sigmoid(out['risk']).squeeze(0).cpu().numpy()
    bert_risks      = [RISK_FACTOR_LABELS[i].replace('_', ' ').title()
                       for i, p in enumerate(risk_probs) if p > risk_threshold]

    # Layer 2: Rule engine (regex on raw text)
    rule_vec        = extract_risk_factors(symptom_text)
    rule_risks      = [RISK_FACTOR_LABELS[i].replace('_', ' ').title()
                       for i, v in enumerate(rule_vec) if v == 1]

    # Layer 3: Knowledge map linked from top diseases
    top_names       = [d['disease'] for d in top_diseases]
    linked_risks    = get_linked_risks(top_names)

    # Merge and deduplicate (preserve insertion order, rule > bert > linked)
    seen, all_risks = set(), []
    for r in rule_risks + bert_risks:
        key = r.lower()
        if key not in seen: seen.add(key); all_risks.append(f'{r}  (from your text)')
    for r in linked_risks:
        key = r.lower()
        if key not in seen: seen.add(key); all_risks.append(f'{r}  (linked to condition)')

    result = {
        'specialist':   pred_specialist,
        'urgency':      pred_urgency,
        'severity':     pred_severity,
        'diseases':     top_diseases,
        'risk_factors': all_risks,
    }

    # ── Pretty print ──────────────────────────────────────────────────────────
    if verbose:
        urgency_emoji = {
            'immediate': '🔴', 'within_24_hours': '🟠',
            'within_a_week': '🟡', 'routine': '🟢'
        }
        print('\n' + '━'*56)
        print('  SWASTHYA CLINICAL ASSESSMENT')
        print('━'*56)
        print(f'  Input     : {symptom_text[:60]}')
        print(f'━'*56)
        print(f'  Specialist: {pred_specialist}')
        print(f'  Urgency   : {urgency_emoji.get(pred_urgency,"")} {pred_urgency}')
        print(f'  Severity  : {pred_severity:.1f} / 10')
        print()
        print('  Possible Conditions:')
        bar_chars = '█░'
        for i, d in enumerate(top_diseases, 1):
            conf  = d["confidence"]
            bar   = int(conf * 10) * bar_chars[0] + (10 - int(conf * 10)) * bar_chars[1]
            print(f'    {i}. {d["disease"]:<35} {bar}  {conf*100:.0f}%')
        print()
        print('  Risk Factors:')
        if all_risks:
            for r in all_risks[:8]:
                print(f'    • {r}')
            if len(all_risks) > 8:
                print(f'    ... and {len(all_risks)-8} more')
        else:
            print('    • No specific risk factors detected')
        print('━'*56)
        print('  ⚠  Clinical support tool — not a medical diagnosis.')
        print('━'*56)

    return result


# ── Run test cases ────────────────────────────────────────────────────────────
test_cases = [
    '55M | chest pain radiating to left arm, sweating, smoker, father had heart attack, BP 160/100',
    '28F | fever 3 days, stomach pain, nausea, vomiting, travel to rural area last week',
    '35M | severe headache, neck stiffness, sensitivity to light since yesterday',
    '45M | frequent urination, excessive thirst, unexplained weight loss, family history of diabetes',
    '30F | anxiety, panic attacks, difficulty sleeping, high work pressure',
    '60M | shortness of breath, wheezing, cough with mucus, chronic smoker',
    '22F | mild skin rash on arms, itching for 2 days after using new soap',
]

print('Running full inference on 7 test cases...')
results = []
for text in test_cases:
    r = predict_full(text, top_k=3, verbose=True)
    results.append(r)

print(f'\nAll {len(results)} test cases completed.')
print('Model is ready for integration into the Swasthya app.')


---
# Part 2 — T5 CoT Question Generation Training

Trains T5 to generate diagnostic follow-up questions with chain-of-thought reasoning.
Questions are extracted from real doctor responses in HealthCareMagic.
The THINK block is rule-based scaffolding — T5 learns to generate QUESTIONS, not reasoning.

**Can run in a separate session** — independent of Part 1.

## Cell 15 — T5 Imports & Config

In [ ]:
import os, re, json
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import T5ForConditionalGeneration, AutoTokenizer
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
from datasets import load_dataset
from tqdm import tqdm

T5_MODEL   = 't5-base'
DRIVE_DIR  = '/kaggle/working'
PAIRS_FILE = os.path.join(DRIVE_DIR, 't5_cot_pairs.json')
device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}  Model : {T5_MODEL}')

## Cell 16 — CoT Reasoning Builder & Question Extractor

**THINK block** — teaches T5 the diagnostic reasoning behind each question.
Example: 'chest or cardiac symptoms detected | need to rule out MI vs angina | ask radiation'
This block is stripped at inference — patients only see the QUESTIONS.

**Question extractor** — pulls real diagnostic questions from doctor responses.
Filters out greetings, social phrases, and non-diagnostic questions.

In [ ]:
def _detect_groups_t5(text):
    t = text.lower()
    return {
        'cardiac':     any(w in t for w in ['chest pain','heart','palpitation','arrhythmia','angina','cardiac']),
        'respiratory': any(w in t for w in ['breath','cough','wheez','lung','pneumonia','asthma','tuberculosis']),
        'neuro':       any(w in t for w in ['headache','migraine','seizure','paralysis','stroke','numbness','dizziness','confusion','fainting','vision']),
        'gastro':      any(w in t for w in ['stomach','abdominal','nausea','vomit','diarrhea','constipation','bloating','heartburn','bowel','liver']),
        'fever':       any(w in t for w in ['fever','chills','sweating','temperature']),
        'ortho':       any(w in t for w in ['joint','back pain','knee','bone','fracture','arthritis','muscle','sprain','shoulder']),
        'skin':        any(w in t for w in ['rash','skin','itch','eczema','psoriasis','hives','acne']),
        'urinary':     any(w in t for w in ['urin','kidney','bladder','burning urination','frequent urination']),
        'mental':      any(w in t for w in ['anxiety','depression','panic','stress','mental','mood','sleep','insomnia']),
        'gynec':       any(w in t for w in ['menstrual','period','vaginal','pregnancy','pelvic','ovary']),
    }

def build_cot_reasoning(text):
    g, cot = _detect_groups_t5(text), []
    if g['cardiac']:     cot += ['chest or cardiac symptoms detected','need to rule out MI vs angina vs musculoskeletal','ask about radiation, duration, associated diaphoresis and dyspnea']
    if g['respiratory']: cot += ['respiratory symptoms detected','need to differentiate infection vs obstruction vs embolism','ask about sputum, fever, onset, and exertional component']
    if g['neuro']:       cot += ['neurological symptoms detected','need to rule out meningitis, stroke, migraine, raised ICP','ask about onset speed, neck stiffness, photophobia, focal deficits']
    if g['gastro'] and g['fever']: cot += ['fever with GI symptoms detected','need to rule out appendicitis, typhoid, cholecystitis','ask about pain location, duration, travel history, bowel changes']
    elif g['gastro']:    cot += ['gastrointestinal symptoms detected','need to differentiate peptic ulcer, GERD, IBD, IBS','ask about relation to meals, blood in stool, weight loss']
    if g['fever'] and not g['gastro'] and not g['respiratory']: cot += ['isolated fever detected','need to rule out viral, bacterial, malaria, typhoid','ask about duration, pattern, travel, contact history']
    if g['ortho']:       cot += ['musculoskeletal symptoms detected','need to differentiate trauma, arthritis, infection, referred pain','ask about injury history, swelling, morning stiffness, activity limitation']
    if g['mental']:      cot += ['mental health symptoms detected','need to assess severity, triggers, functional impact','ask about duration, sleep, appetite, suicidal ideation, triggers']
    if g['urinary']:     cot += ['urinary symptoms detected','need to rule out UTI, kidney stones, diabetes, prostate issues','ask about burning, frequency, blood in urine, flank pain']
    if g['skin']:        cot += ['skin symptoms detected','need to differentiate allergic, infectious, autoimmune','ask about distribution, triggers, associated fever, new products']
    if g['gynec']:       cot += ['gynecological symptoms detected','need to assess cycle regularity, pregnancy possibility, infection','ask about last period, discharge, pain character']
    if not cot:          cot  = ['general symptoms detected','need to assess duration, severity, and associated features','ask about onset, progression, and impact on daily activities']
    return ' | '.join(cot)

SKIP_PHRASES = [
    'how are you', 'how do you do', 'how can i help', 'any other',
    'anything else', 'is there anything', 'hope i have', 'not to worry',
    'consult a doctor', 'i would suggest', 'i would advise', 'dear friend',
    'thanks for', 'chat doctor', 'i understand', 'i studied your'
]

def extract_questions(doctor_text):
    """Extract real diagnostic questions from doctor response text.
    Filters: greetings, social phrases, too short/long, not ending with '?'"""
    questions = []
    for s in re.split(r'(?<=[.!?])\s+', doctor_text.strip()):
        s = s.strip()
        if not s.endswith('?') or len(s) < 15 or len(s) > 200: continue
        if any(p in s.lower() for p in SKIP_PHRASES): continue
        questions.append(s)
    return questions[:5]

print('T5 CoT builder loaded.')

## Cell 17 — Build CoT Training Pairs
Combines patient text + CoT reasoning + real doctor questions into training pairs.
Cached to storage after first run.

Input:  `'symptoms: <patient text>'`
Target: `'THINK: <reasoning> QUESTIONS: Q1? [SEP] Q2? [SEP] Q3?'`

In [ ]:
def build_cot_pairs(dataset_split, max_samples=60000):
    pairs, skipped = [], 0
    for item in tqdm(dataset_split, desc='Building CoT pairs'):
        pt = str(item.get('input',  '')).strip()
        dt = str(item.get('output', '')).strip()
        if len(pt) < 20 or not dt: skipped += 1; continue
        qs = extract_questions(dt)
        if len(qs) < 2: skipped += 1; continue
        pairs.append({'input':  f'symptoms: {pt[:400]}',
                      'target': f'THINK: {build_cot_reasoning(pt)} QUESTIONS: {" [SEP] ".join(qs)}'})
        if len(pairs) >= max_samples: break
    print(f'Built {len(pairs):,} pairs  |  skipped {skipped:,}')
    return pairs

if os.path.exists(PAIRS_FILE):
    print('Loading cached CoT pairs...')
    with open(PAIRS_FILE) as f: all_pairs = json.load(f)
    print(f'Loaded {len(all_pairs):,} pairs')
else:
    print('Downloading HealthCareMagic-100k for T5...')
    ds = load_dataset('lavita/ChatDoctor-HealthCareMagic-100k')['train']
    all_pairs = build_cot_pairs(ds, max_samples=60000)
    with open(PAIRS_FILE, 'w') as f: json.dump(all_pairs, f)
    print(f'Saved {len(all_pairs):,} pairs to {PAIRS_FILE}')

print(f'\nSample:')
print(f'  IN : {all_pairs[0]["input"][:80]}')
print(f'  OUT: {all_pairs[0]["target"][:120]}')
split = int(len(all_pairs) * 0.9)
train_pairs, val_pairs = all_pairs[:split], all_pairs[split:]
print(f'\nTrain: {len(train_pairs):,}  Val: {len(val_pairs):,}')

## Cell 18 — T5 Dataset & Model
Loads t5-base (~250M params) or resumes from a previous checkpoint.

In [ ]:
class T5CotDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_input=256, max_target=200):
        self.pairs = pairs; self.tokenizer = tokenizer
        self.max_input = max_input; self.max_target = max_target
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        pair = self.pairs[idx]
        inp  = self.tokenizer(pair['input'],  max_length=self.max_input,  padding='max_length', truncation=True, return_tensors='pt')
        tgt  = self.tokenizer(pair['target'], max_length=self.max_target, padding='max_length', truncation=True, return_tensors='pt')
        labels = tgt['input_ids'].squeeze(0).clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        return {'input_ids': inp['input_ids'].squeeze(0),
                'attention_mask': inp['attention_mask'].squeeze(0),
                'labels': labels}

T5_CKPT = os.path.join(DRIVE_DIR, 't5_cot_checkpoint')
if os.path.exists(T5_CKPT):
    print('Resuming T5 from checkpoint...')
    t5_tok = AutoTokenizer.from_pretrained(T5_CKPT)
    t5_mdl = T5ForConditionalGeneration.from_pretrained(T5_CKPT).to(device)
else:
    print(f'Loading fresh {T5_MODEL}...')
    t5_tok = AutoTokenizer.from_pretrained(T5_MODEL)
    t5_mdl = T5ForConditionalGeneration.from_pretrained(T5_MODEL).to(device)

t5_train_loader = DataLoader(T5CotDataset(train_pairs, t5_tok), batch_size=8,  shuffle=True,  num_workers=2, pin_memory=True)
t5_val_loader   = DataLoader(T5CotDataset(val_pairs,   t5_tok), batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
print(f'T5 params     : {sum(p.numel() for p in t5_mdl.parameters()):,}')
print(f'Train batches : {len(t5_train_loader):,}  Val batches: {len(t5_val_loader):,}')

## Cell 19 — Train T5
Saves checkpoint every epoch — auto-resumes if session drops.
Best model saved to `t5_qgen_model/` whenever val_loss improves.

**Target val_loss: ~1.2-1.5**  
⏱️ ~2 hrs

In [ ]:
NUM_EPOCHS_T5 = 5; LR_T5 = 3e-4; PATIENCE_T5 = 3
STATE_FILE = os.path.join(DRIVE_DIR, 't5_train_state.json')
start_epoch = 0; best_val_loss = float('inf'); patience_count = 0; t5_history = []

if os.path.exists(STATE_FILE):
    with open(STATE_FILE) as f: state = json.load(f)
    start_epoch    = state['epoch'] + 1
    best_val_loss  = state['best_val_loss']
    patience_count = state['patience_count']
    t5_history     = state['history']
    print(f'Resuming T5 from epoch {start_epoch}, best_val_loss={best_val_loss:.4f}')

t5_opt = AdamW(t5_mdl.parameters(), lr=LR_T5, weight_decay=0.01)
t5_sch = get_linear_schedule_with_warmup(
    t5_opt,
    num_warmup_steps=int(len(t5_train_loader) * NUM_EPOCHS_T5 * 0.05),
    num_training_steps=len(t5_train_loader) * NUM_EPOCHS_T5
)

for epoch in range(start_epoch, NUM_EPOCHS_T5):
    t5_mdl.train(); train_loss = 0.0
    for b in tqdm(t5_train_loader, desc=f'T5 E{epoch+1}/{NUM_EPOCHS_T5}'):
        out = t5_mdl(input_ids=b['input_ids'].to(device),
                     attention_mask=b['attention_mask'].to(device),
                     labels=b['labels'].to(device))
        t5_opt.zero_grad(); out.loss.backward()
        torch.nn.utils.clip_grad_norm_(t5_mdl.parameters(), 1.0)
        t5_opt.step(); t5_sch.step(); train_loss += out.loss.item()
    train_loss /= len(t5_train_loader)

    t5_mdl.eval(); val_loss = 0.0
    with torch.no_grad():
        for b in tqdm(t5_val_loader, desc='Eval', leave=False):
            out = t5_mdl(input_ids=b['input_ids'].to(device),
                         attention_mask=b['attention_mask'].to(device),
                         labels=b['labels'].to(device))
            val_loss += out.loss.item()
    val_loss /= len(t5_val_loader)
    print(f'Epoch {epoch+1}/{NUM_EPOCHS_T5} | train={train_loss:.4f} | val={val_loss:.4f}')
    t5_history.append({'epoch': epoch+1, 'train_loss': train_loss, 'val_loss': val_loss})

    if val_loss < best_val_loss:
        best_val_loss = val_loss; patience_count = 0
        t5_mdl.save_pretrained(os.path.join(DRIVE_DIR, 't5_qgen_model'))
        t5_tok.save_pretrained(os.path.join(DRIVE_DIR, 't5_qgen_model'))
        print(f'  Saved best (val_loss={best_val_loss:.4f})')
    else:
        patience_count += 1
        if patience_count >= PATIENCE_T5: print('Early stopping.'); break

    t5_mdl.save_pretrained(T5_CKPT); t5_tok.save_pretrained(T5_CKPT)
    with open(STATE_FILE, 'w') as f:
        json.dump({'epoch': epoch, 'best_val_loss': best_val_loss,
                   'patience_count': patience_count, 'history': t5_history}, f)

print(f'\nT5 done. Best val_loss: {best_val_loss:.4f}  (target ~1.2-1.5)')

## Cell 20 — T5 Inference Test
Verifies the T5 model generates targeted questions, not generic ones.
Reasoning shown for each case — confirms the CoT mechanism is working.

In [ ]:
t5_mdl = T5ForConditionalGeneration.from_pretrained(
    os.path.join(DRIVE_DIR, 't5_qgen_model')).to(device)
t5_mdl.eval()

def generate_questions(symptom_text, show_reasoning=False):
    """Generate diagnostic questions. THINK block stripped — patients see QUESTIONS only."""
    ids = t5_tok(f'symptoms: {symptom_text}', return_tensors='pt',
                 max_length=256, truncation=True).input_ids.to(device)
    with torch.no_grad():
        out = t5_mdl.generate(ids, max_new_tokens=200, num_beams=4,
                               early_stopping=True, no_repeat_ngram_size=3)
    decoded = t5_tok.decode(out[0], skip_special_tokens=True)
    if show_reasoning and 'THINK:' in decoded:
        think = decoded.split('QUESTIONS:')[0].replace('THINK:', '').strip()
        print(f'  Reasoning: {think[:120]}')
    q_part    = decoded.split('QUESTIONS:')[-1].strip() if 'QUESTIONS:' in decoded else decoded
    questions = [q.strip() for q in q_part.split('[SEP]') if len(q.strip()) > 10]
    return [q if q.endswith('?') else q+'?' for q in questions][:5]

tests = [
    'I have chest pain radiating to my left arm and sweating',
    'I have had fever for 3 days with stomach pain and vomiting',
    'I have severe headache and neck stiffness since yesterday',
    'I feel very anxious and have panic attacks at night',
    'My knee is swollen and painful after a fall',
]
print('T5 inference test (reasoning shown):')
print('-'*65)
for s in tests:
    print(f'Symptom: {s}')
    for i, q in enumerate(generate_questions(s, show_reasoning=True), 1):
        print(f'  Q{i}: {q}')
    print()

## Cell 21 — Training Curves + Final Download Summary

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if all_history:
    ep = [h['epoch'] for h in all_history]
    axes[0].plot(ep, [h['specialist_acc'] for h in all_history], marker='o', label='Specialist')
    axes[0].plot(ep, [h['urgency_acc']    for h in all_history], marker='s', label='Urgency')
    axes[0].axhline(y=0.90, color='green', linestyle='--', alpha=0.5, label='Target 90%')
    axes[0].set_title('Bio_ClinicalBERT (3k/class balanced)')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
    axes[0].legend(); axes[0].grid(alpha=0.3)
else:
    axes[0].text(0.5, 0.5, 'Run Part 1 first', ha='center', va='center', transform=axes[0].transAxes)

if 't5_history' in dir() and t5_history:
    ep = [h['epoch'] for h in t5_history]
    axes[1].plot(ep, [h['train_loss'] for h in t5_history], marker='o', label='Train')
    axes[1].plot(ep, [h['val_loss']   for h in t5_history], marker='s', label='Val')
    axes[1].axhline(y=1.5, color='green', linestyle='--', alpha=0.5, label='Target ~1.5')
    axes[1].set_title('T5 CoT loss')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
    axes[1].legend(); axes[1].grid(alpha=0.3)
else:
    axes[1].text(0.5, 0.5, 'Run Part 2 first', ha='center', va='center', transform=axes[1].transAxes)

plt.tight_layout()
plt.savefig(os.path.join(DRIVE_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\n' + '='*55)
print('FINAL FILES — download from Kaggle Output tab')
print('Copy all to ml_models/models/ on your machine')
print('='*55)
for fname in ['biobert_clinical_best.pth', 'urgency_encoder.pkl',
              'specialist_encoder.pkl', 'training_history.json']:
    full = os.path.join(DRIVE_DIR, fname)
    status = f'{os.path.getsize(full)/1e6:.1f}MB' if os.path.exists(full) else 'MISSING'
    print(f'  {status:<10} {fname}')
t5_dir = os.path.join(DRIVE_DIR, 't5_qgen_model')
if os.path.exists(t5_dir):
    sz = sum(os.path.getsize(os.path.join(t5_dir, f)) for f in os.listdir(t5_dir)) / 1e6
    print(f'  {sz:.1f}MB     t5_qgen_model/  (entire folder)')
else:
    print('  MISSING    t5_qgen_model/')
print('\nDownload from Kaggle: Output tab (right sidebar) -> click file -> Download')
print('For t5_qgen_model folder: zip it first then download:')
print("  !zip -r /kaggle/working/t5_qgen_model.zip /kaggle/working/t5_qgen_model")